# Femora — Breast Lesion Outline (U-Net)

The classifier says *what* a scan most likely shows; its heatmap shows roughly *where it looked*. This notebook trains a
second, separate model that draws the **outline of the lesion itself**, so the app can show its shape and an approximate size.

**Data**: every scan with a radiologist's lesion mask from BUSI, BrEaST-Lesions-USG and BUS-BRA, plus BUSI's normal scans
(empty masks, so the model learns to draw nothing when there is no lesion).

**Split**: exactly the deployed classifier's train / validation / test assignment (`ml/busbra_split.csv`). The test scans
were never seen by either model, so the outline is judged on the same images as the classifier.

**Model**: U-Net with an ImageNet-pretrained ResNet34 encoder, 256 × 256 grayscale input padded (not stretched) to a square,
the same preprocessing as the classifier. Loss: binary cross-entropy + Dice. The epoch with the best validation Dice is kept.

**Reported on the test set** (with the exported ONNX model, which is what the backend runs): Dice and IoU per hospital, the
share of lesions outlined reasonably well (Dice ≥ 0.5), and how often an outline is wrongly drawn on a normal scan.

In [ ]:
%pip install -q segmentation-models-pytorch onnx onnxruntime onnxscript

In [ ]:
import copy, glob, io, json, os, random, shutil, urllib.request, zipfile, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision import tv_tensors
import segmentation_models_pytorch as smp
warnings.filterwarnings("ignore")

INPUT, WORK, TMP = "/kaggle/input", "/kaggle/working", "/tmp/femora"
OUT = f"{WORK}/model"
for d in (OUT, TMP):
    os.makedirs(d, exist_ok=True)
SEED, IMG, BATCH, EPOCHS = 42, 256, 16, 45
MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.__version__, smp.__version__)

# ---- Preprocessing shared with backend/app.py (keep the two in sync)
def pad_square(img, fill=0):
    w, h = img.size
    s = max(w, h)
    canvas = Image.new(img.mode, (s, s), fill)
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas

def to_gray(img):
    return pad_square(ImageOps.exif_transpose(img).convert("L")).resize((IMG, IMG), Image.BILINEAR)

def to_mask(paths, size):
    # several masks per scan (BUSI) are merged; padded like the scan, nearest-neighbour so it stays binary
    m = np.zeros(size[::-1], bool)
    for p in paths:
        m |= np.asarray(Image.open(p).convert("L").resize(size, Image.NEAREST)) > 127
    return np.asarray(pad_square(Image.fromarray(m.astype(np.uint8) * 255)).resize((IMG, IMG), Image.NEAREST)) > 127

def normalize(gray):
    x = np.asarray(gray, np.float32) / 255.0
    return ((np.repeat(x[None], 3, 0) - MEAN) / STD).astype(np.float32)

def download(url, dest):
    if not os.path.exists(dest):
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
    return dest

In [ ]:
SPLIT_CSV = '''source,file,split
BUSI,normal (10).png,train
BUSI,normal (100).png,train
BUSI,normal (101).png,test
BUSI,normal (102).png,train
BUSI,normal (103).png,test
BUSI,normal (104).png,val
BUSI,normal (105).png,train
BUSI,normal (106).png,test
BUSI,normal (107).png,val
BUSI,normal (108).png,train
BUSI,normal (109).png,train
BUSI,normal (11).png,train
BUSI,normal (110).png,train
BUSI,normal (111).png,train
BUSI,normal (112).png,val
BUSI,normal (113).png,test
BUSI,normal (114).png,val
BUSI,normal (115).png,train
BUSI,normal (116).png,train
BUSI,normal (117).png,train
BUSI,normal (118).png,train
BUSI,normal (119).png,train
BUSI,normal (12).png,test
BUSI,normal (120).png,val
BUSI,normal (121).png,train
BUSI,normal (122).png,train
BUSI,normal (123).png,train
BUSI,normal (124).png,val
BUSI,normal (125).png,train
BUSI,normal (126).png,train
BUSI,normal (127).png,train
BUSI,normal (128).png,train
BUSI,normal (129).png,train
BUSI,normal (13).png,test
BUSI,normal (130).png,test
BUSI,normal (131).png,val
BUSI,normal (132).png,train
BUSI,normal (133).png,test
BUSI,normal (14).png,train
BUSI,normal (15).png,train
BUSI,normal (16).png,train
BUSI,normal (17).png,train
BUSI,normal (18).png,test
BUSI,normal (19).png,train
BUSI,normal (2).png,train
BUSI,normal (20).png,train
BUSI,normal (21).png,val
BUSI,normal (22).png,train
BUSI,normal (23).png,train
BUSI,normal (24).png,train
BUSI,normal (25).png,train
BUSI,normal (26).png,test
BUSI,normal (27).png,val
BUSI,normal (28).png,train
BUSI,normal (29).png,train
BUSI,normal (3).png,train
BUSI,normal (30).png,train
BUSI,normal (31).png,train
BUSI,normal (32).png,train
BUSI,normal (33).png,train
BUSI,normal (35).png,train
BUSI,normal (36).png,train
BUSI,normal (37).png,test
BUSI,normal (38).png,val
BUSI,normal (39).png,train
BUSI,normal (4).png,train
BUSI,normal (40).png,val
BUSI,normal (41).png,train
BUSI,normal (42).png,train
BUSI,normal (43).png,train
BUSI,normal (44).png,train
BUSI,normal (45).png,train
BUSI,normal (46).png,val
BUSI,normal (47).png,train
BUSI,normal (48).png,train
BUSI,normal (49).png,train
BUSI,normal (5).png,test
BUSI,normal (50).png,test
BUSI,normal (51).png,train
BUSI,normal (52).png,train
BUSI,normal (53).png,train
BUSI,normal (54).png,train
BUSI,normal (55).png,train
BUSI,normal (56).png,val
BUSI,normal (57).png,train
BUSI,normal (58).png,train
BUSI,normal (59).png,test
BUSI,normal (6).png,train
BUSI,normal (60).png,train
BUSI,normal (61).png,train
BUSI,normal (62).png,train
BUSI,normal (63).png,train
BUSI,normal (64).png,train
BUSI,normal (65).png,train
BUSI,normal (66).png,train
BUSI,normal (67).png,train
BUSI,normal (68).png,test
BUSI,normal (69).png,train
BUSI,normal (7).png,val
BUSI,normal (70).png,train
BUSI,normal (71).png,train
BUSI,normal (72).png,train
BUSI,normal (73).png,test
BUSI,normal (74).png,train
BUSI,normal (75).png,train
BUSI,normal (76).png,val
BUSI,normal (77).png,train
BUSI,normal (78).png,train
BUSI,normal (79).png,train
BUSI,normal (8).png,train
BUSI,normal (80).png,train
BUSI,normal (81).png,test
BUSI,normal (82).png,train
BUSI,normal (83).png,train
BUSI,normal (84).png,train
BUSI,normal (85).png,val
BUSI,normal (86).png,test
BUSI,normal (87).png,train
BUSI,normal (88).png,train
BUSI,normal (89).png,train
BUSI,normal (9).png,test
BUSI,normal (90).png,train
BUSI,normal (91).png,val
BUSI,normal (92).png,val
BUSI,normal (93).png,train
BUSI,normal (94).png,train
BUSI,normal (95).png,train
BUSI,normal (96).png,train
BUSI,normal (97).png,test
BUSI,normal (98).png,test
BUSI,normal (99).png,val
BUSI,benign (1).png,train
BUSI,benign (10).png,train
BUSI,benign (100).png,train
BUSI,benign (101).png,train
BUSI,benign (102).png,train
BUSI,benign (103).png,train
BUSI,benign (104).png,train
BUSI,benign (105).png,train
BUSI,benign (106).png,test
BUSI,benign (107).png,train
BUSI,benign (108).png,test
BUSI,benign (109).png,train
BUSI,benign (11).png,train
BUSI,benign (110).png,val
BUSI,benign (111).png,train
BUSI,benign (112).png,train
BUSI,benign (113).png,train
BUSI,benign (114).png,train
BUSI,benign (115).png,test
BUSI,benign (116).png,val
BUSI,benign (117).png,train
BUSI,benign (118).png,train
BUSI,benign (119).png,train
BUSI,benign (12).png,val
BUSI,benign (120).png,train
BUSI,benign (121).png,test
BUSI,benign (122).png,train
BUSI,benign (123).png,train
BUSI,benign (124).png,val
BUSI,benign (125).png,train
BUSI,benign (126).png,train
BUSI,benign (127).png,train
BUSI,benign (128).png,train
BUSI,benign (129).png,train
BUSI,benign (13).png,train
BUSI,benign (130).png,test
BUSI,benign (132).png,val
BUSI,benign (133).png,train
BUSI,benign (134).png,train
BUSI,benign (135).png,train
BUSI,benign (136).png,train
BUSI,benign (137).png,train
BUSI,benign (138).png,val
BUSI,benign (139).png,train
BUSI,benign (14).png,train
BUSI,benign (140).png,test
BUSI,benign (141).png,train
BUSI,benign (142).png,train
BUSI,benign (143).png,val
BUSI,benign (144).png,train
BUSI,benign (145).png,train
BUSI,benign (146).png,train
BUSI,benign (147).png,train
BUSI,benign (148).png,train
BUSI,benign (149).png,val
BUSI,benign (15).png,val
BUSI,benign (150).png,train
BUSI,benign (151).png,train
BUSI,benign (152).png,train
BUSI,benign (153).png,val
BUSI,benign (154).png,test
BUSI,benign (155).png,train
BUSI,benign (156).png,train
BUSI,benign (157).png,train
BUSI,benign (158).png,train
BUSI,benign (159).png,train
BUSI,benign (16).png,test
BUSI,benign (160).png,train
BUSI,benign (161).png,val
BUSI,benign (162).png,train
BUSI,benign (163).png,train
BUSI,benign (165).png,train
BUSI,benign (166).png,train
BUSI,benign (167).png,train
BUSI,benign (168).png,train
BUSI,benign (169).png,test
BUSI,benign (17).png,train
BUSI,benign (170).png,val
BUSI,benign (171).png,train
BUSI,benign (172).png,test
BUSI,benign (173).png,val
BUSI,benign (174).png,train
BUSI,benign (175).png,train
BUSI,benign (176).png,train
BUSI,benign (177).png,test
BUSI,benign (178).png,train
BUSI,benign (179).png,val
BUSI,benign (18).png,train
BUSI,benign (180).png,train
BUSI,benign (181).png,test
BUSI,benign (182).png,train
BUSI,benign (183).png,train
BUSI,benign (184).png,train
BUSI,benign (185).png,train
BUSI,benign (186).png,train
BUSI,benign (187).png,train
BUSI,benign (188).png,train
BUSI,benign (189).png,train
BUSI,benign (19).png,train
BUSI,benign (190).png,train
BUSI,benign (191).png,test
BUSI,benign (192).png,val
BUSI,benign (193).png,test
BUSI,benign (194).png,train
BUSI,benign (195).png,train
BUSI,benign (196).png,train
BUSI,benign (197).png,train
BUSI,benign (198).png,train
BUSI,benign (199).png,test
BUSI,benign (2).png,train
BUSI,benign (20).png,val
BUSI,benign (200).png,train
BUSI,benign (201).png,train
BUSI,benign (202).png,train
BUSI,benign (203).png,train
BUSI,benign (204).png,train
BUSI,benign (205).png,test
BUSI,benign (206).png,train
BUSI,benign (207).png,val
BUSI,benign (208).png,train
BUSI,benign (209).png,train
BUSI,benign (21).png,train
BUSI,benign (210).png,test
BUSI,benign (211).png,val
BUSI,benign (212).png,train
BUSI,benign (213).png,train
BUSI,benign (214).png,train
BUSI,benign (215).png,train
BUSI,benign (216).png,train
BUSI,benign (217).png,val
BUSI,benign (218).png,train
BUSI,benign (219).png,test
BUSI,benign (22).png,train
BUSI,benign (220).png,train
BUSI,benign (221).png,train
BUSI,benign (222).png,val
BUSI,benign (223).png,train
BUSI,benign (224).png,train
BUSI,benign (225).png,train
BUSI,benign (226).png,val
BUSI,benign (227).png,train
BUSI,benign (228).png,train
BUSI,benign (229).png,train
BUSI,benign (23).png,train
BUSI,benign (230).png,test
BUSI,benign (231).png,train
BUSI,benign (232).png,val
BUSI,benign (233).png,val
BUSI,benign (234).png,train
BUSI,benign (235).png,train
BUSI,benign (236).png,train
BUSI,benign (237).png,val
BUSI,benign (238).png,train
BUSI,benign (239).png,train
BUSI,benign (24).png,test
BUSI,benign (240).png,test
BUSI,benign (241).png,train
BUSI,benign (242).png,train
BUSI,benign (243).png,train
BUSI,benign (244).png,train
BUSI,benign (245).png,train
BUSI,benign (246).png,train
BUSI,benign (247).png,train
BUSI,benign (248).png,train
BUSI,benign (249).png,train
BUSI,benign (25).png,val
BUSI,benign (250).png,train
BUSI,benign (251).png,train
BUSI,benign (252).png,train
BUSI,benign (253).png,train
BUSI,benign (254).png,train
BUSI,benign (255).png,test
BUSI,benign (256).png,test
BUSI,benign (257).png,val
BUSI,benign (258).png,train
BUSI,benign (259).png,train
BUSI,benign (26).png,train
BUSI,benign (260).png,train
BUSI,benign (261).png,train
BUSI,benign (262).png,train
BUSI,benign (263).png,val
BUSI,benign (264).png,train
BUSI,benign (265).png,train
BUSI,benign (266).png,train
BUSI,benign (267).png,test
BUSI,benign (268).png,val
BUSI,benign (27).png,train
BUSI,benign (270).png,train
BUSI,benign (271).png,train
BUSI,benign (272).png,train
BUSI,benign (273).png,train
BUSI,benign (274).png,test
BUSI,benign (275).png,train
BUSI,benign (276).png,val
BUSI,benign (277).png,test
BUSI,benign (278).png,val
BUSI,benign (279).png,train
BUSI,benign (28).png,train
BUSI,benign (280).png,train
BUSI,benign (281).png,train
BUSI,benign (282).png,train
BUSI,benign (283).png,train
BUSI,benign (284).png,test
BUSI,benign (285).png,test
BUSI,benign (286).png,train
BUSI,benign (287).png,val
BUSI,benign (288).png,train
BUSI,benign (289).png,train
BUSI,benign (29).png,train
BUSI,benign (290).png,test
BUSI,benign (291).png,val
BUSI,benign (292).png,train
BUSI,benign (293).png,train
BUSI,benign (294).png,train
BUSI,benign (295).png,train
BUSI,benign (296).png,train
BUSI,benign (297).png,train
BUSI,benign (298).png,test
BUSI,benign (299).png,val
BUSI,benign (3).png,test
BUSI,benign (30).png,train
BUSI,benign (300).png,train
BUSI,benign (301).png,train
BUSI,benign (302).png,val
BUSI,benign (303).png,train
BUSI,benign (304).png,train
BUSI,benign (305).png,train
BUSI,benign (306).png,train
BUSI,benign (307).png,train
BUSI,benign (308).png,train
BUSI,benign (309).png,train
BUSI,benign (31).png,train
BUSI,benign (310).png,test
BUSI,benign (311).png,train
BUSI,benign (312).png,val
BUSI,benign (313).png,train
BUSI,benign (314).png,train
BUSI,benign (315).png,test
BUSI,benign (316).png,test
BUSI,benign (317).png,val
BUSI,benign (318).png,train
BUSI,benign (319).png,train
BUSI,benign (32).png,test
BUSI,benign (320).png,test
BUSI,benign (321).png,train
BUSI,benign (322).png,train
BUSI,benign (323).png,train
BUSI,benign (324).png,train
BUSI,benign (325).png,train
BUSI,benign (326).png,train
BUSI,benign (327).png,train
BUSI,benign (328).png,val
BUSI,benign (329).png,train
BUSI,benign (33).png,test
BUSI,benign (330).png,train
BUSI,benign (331).png,train
BUSI,benign (332).png,val
BUSI,benign (333).png,train
BUSI,benign (334).png,train
BUSI,benign (335).png,test
BUSI,benign (336).png,val
BUSI,benign (337).png,train
BUSI,benign (338).png,train
BUSI,benign (339).png,train
BUSI,benign (34).png,train
BUSI,benign (340).png,train
BUSI,benign (341).png,test
BUSI,benign (342).png,val
BUSI,benign (343).png,train
BUSI,benign (344).png,train
BUSI,benign (345).png,val
BUSI,benign (346).png,train
BUSI,benign (347).png,train
BUSI,benign (348).png,train
BUSI,benign (349).png,train
BUSI,benign (35).png,test
BUSI,benign (350).png,train
BUSI,benign (351).png,train
BUSI,benign (352).png,train
BUSI,benign (353).png,val
BUSI,benign (354).png,train
BUSI,benign (355).png,test
BUSI,benign (356).png,train
BUSI,benign (357).png,val
BUSI,benign (358).png,train
BUSI,benign (359).png,train
BUSI,benign (36).png,train
BUSI,benign (360).png,train
BUSI,benign (361).png,train
BUSI,benign (362).png,test
BUSI,benign (363).png,train
BUSI,benign (364).png,val
BUSI,benign (365).png,train
BUSI,benign (366).png,train
BUSI,benign (367).png,train
BUSI,benign (368).png,train
BUSI,benign (369).png,train
BUSI,benign (37).png,train
BUSI,benign (370).png,test
BUSI,benign (371).png,val
BUSI,benign (372).png,train
BUSI,benign (373).png,train
BUSI,benign (374).png,train
BUSI,benign (375).png,train
BUSI,benign (376).png,test
BUSI,benign (377).png,train
BUSI,benign (378).png,train
BUSI,benign (379).png,train
BUSI,benign (38).png,val
BUSI,benign (380).png,train
BUSI,benign (381).png,train
BUSI,benign (382).png,train
BUSI,benign (383).png,val
BUSI,benign (384).png,train
BUSI,benign (385).png,test
BUSI,benign (386).png,train
BUSI,benign (387).png,train
BUSI,benign (388).png,train
BUSI,benign (389).png,train
BUSI,benign (39).png,train
BUSI,benign (390).png,train
BUSI,benign (391).png,train
BUSI,benign (392).png,train
BUSI,benign (393).png,test
BUSI,benign (394).png,train
BUSI,benign (395).png,test
BUSI,benign (396).png,train
BUSI,benign (397).png,train
BUSI,benign (398).png,val
BUSI,benign (4).png,train
BUSI,benign (40).png,train
BUSI,benign (400).png,test
BUSI,benign (401).png,train
BUSI,benign (402).png,train
BUSI,benign (403).png,test
BUSI,benign (404).png,val
BUSI,benign (405).png,train
BUSI,benign (406).png,train
BUSI,benign (407).png,train
BUSI,benign (408).png,train
BUSI,benign (409).png,train
BUSI,benign (41).png,train
BUSI,benign (410).png,train
BUSI,benign (411).png,test
BUSI,benign (412).png,test
BUSI,benign (413).png,val
BUSI,benign (414).png,train
BUSI,benign (415).png,train
BUSI,benign (416).png,train
BUSI,benign (417).png,train
BUSI,benign (418).png,val
BUSI,benign (419).png,train
BUSI,benign (420).png,test
BUSI,benign (421).png,val
BUSI,benign (422).png,train
BUSI,benign (423).png,train
BUSI,benign (424).png,train
BUSI,benign (425).png,train
BUSI,benign (426).png,test
BUSI,benign (427).png,train
BUSI,benign (428).png,train
BUSI,benign (429).png,val
BUSI,benign (43).png,train
BUSI,benign (430).png,val
BUSI,benign (431).png,train
BUSI,benign (432).png,train
BUSI,benign (434).png,train
BUSI,benign (435).png,train
BUSI,benign (436).png,train
BUSI,benign (44).png,test
BUSI,benign (45).png,train
BUSI,benign (46).png,val
BUSI,benign (47).png,train
BUSI,benign (48).png,train
BUSI,benign (49).png,train
BUSI,benign (5).png,train
BUSI,benign (50).png,test
BUSI,benign (51).png,train
BUSI,benign (52).png,train
BUSI,benign (53).png,train
BUSI,benign (54).png,train
BUSI,benign (55).png,val
BUSI,benign (56).png,train
BUSI,benign (57).png,train
BUSI,benign (58).png,train
BUSI,benign (59).png,test
BUSI,benign (6).png,train
BUSI,benign (60).png,val
BUSI,benign (61).png,train
BUSI,benign (62).png,train
BUSI,benign (63).png,train
BUSI,benign (64).png,val
BUSI,benign (65).png,train
BUSI,benign (66).png,train
BUSI,benign (67).png,test
BUSI,benign (68).png,train
BUSI,benign (69).png,train
BUSI,benign (7).png,train
BUSI,benign (70).png,train
BUSI,benign (71).png,train
BUSI,benign (72).png,test
BUSI,benign (73).png,train
BUSI,benign (74).png,train
BUSI,benign (75).png,val
BUSI,benign (76).png,train
BUSI,benign (77).png,train
BUSI,benign (78).png,val
BUSI,benign (79).png,train
BUSI,benign (8).png,train
BUSI,benign (80).png,train
BUSI,benign (81).png,test
BUSI,benign (82).png,train
BUSI,benign (83).png,train
BUSI,benign (84).png,train
BUSI,benign (86).png,train
BUSI,benign (87).png,test
BUSI,benign (88).png,val
BUSI,benign (89).png,train
BUSI,benign (9).png,train
BUSI,benign (90).png,train
BUSI,benign (91).png,train
BUSI,benign (92).png,train
BUSI,benign (93).png,train
BUSI,benign (94).png,test
BUSI,benign (95).png,val
BUSI,benign (96).png,train
BUSI,benign (97).png,train
BUSI,benign (98).png,train
BUSI,benign (99).png,train
BUSI,malignant (1).png,train
BUSI,malignant (10).png,val
BUSI,malignant (100).png,train
BUSI,malignant (101).png,train
BUSI,malignant (102).png,test
BUSI,malignant (103).png,train
BUSI,malignant (104).png,test
BUSI,malignant (105).png,train
BUSI,malignant (106).png,train
BUSI,malignant (107).png,train
BUSI,malignant (108).png,test
BUSI,malignant (109).png,test
BUSI,malignant (11).png,val
BUSI,malignant (110).png,train
BUSI,malignant (111).png,val
BUSI,malignant (112).png,train
BUSI,malignant (113).png,train
BUSI,malignant (114).png,train
BUSI,malignant (115).png,val
BUSI,malignant (116).png,train
BUSI,malignant (117).png,train
BUSI,malignant (118).png,train
BUSI,malignant (119).png,train
BUSI,malignant (12).png,train
BUSI,malignant (120).png,test
BUSI,malignant (121).png,train
BUSI,malignant (122).png,val
BUSI,malignant (123).png,train
BUSI,malignant (124).png,train
BUSI,malignant (125).png,train
BUSI,malignant (126).png,train
BUSI,malignant (127).png,train
BUSI,malignant (128).png,train
BUSI,malignant (129).png,train
BUSI,malignant (13).png,test
BUSI,malignant (130).png,train
BUSI,malignant (131).png,val
BUSI,malignant (132).png,train
BUSI,malignant (133).png,test
BUSI,malignant (134).png,train
BUSI,malignant (135).png,train
BUSI,malignant (136).png,train
BUSI,malignant (137).png,train
BUSI,malignant (138).png,train
BUSI,malignant (139).png,train
BUSI,malignant (14).png,val
BUSI,malignant (140).png,train
BUSI,malignant (141).png,train
BUSI,malignant (142).png,test
BUSI,malignant (143).png,train
BUSI,malignant (144).png,train
BUSI,malignant (146).png,val
BUSI,malignant (147).png,train
BUSI,malignant (148).png,train
BUSI,malignant (149).png,test
BUSI,malignant (15).png,train
BUSI,malignant (150).png,train
BUSI,malignant (151).png,train
BUSI,malignant (152).png,train
BUSI,malignant (153).png,test
BUSI,malignant (154).png,train
BUSI,malignant (155).png,val
BUSI,malignant (156).png,train
BUSI,malignant (157).png,train
BUSI,malignant (158).png,train
BUSI,malignant (159).png,train
BUSI,malignant (16).png,train
BUSI,malignant (160).png,train
BUSI,malignant (161).png,train
BUSI,malignant (162).png,test
BUSI,malignant (163).png,train
BUSI,malignant (164).png,val
BUSI,malignant (165).png,train
BUSI,malignant (166).png,train
BUSI,malignant (167).png,val
BUSI,malignant (168).png,train
BUSI,malignant (169).png,train
BUSI,malignant (17).png,test
BUSI,malignant (170).png,val
BUSI,malignant (171).png,train
BUSI,malignant (172).png,train
BUSI,malignant (173).png,train
BUSI,malignant (174).png,test
BUSI,malignant (175).png,train
BUSI,malignant (176).png,train
BUSI,malignant (177).png,train
BUSI,malignant (178).png,val
BUSI,malignant (179).png,train
BUSI,malignant (18).png,train
BUSI,malignant (180).png,train
BUSI,malignant (181).png,train
BUSI,malignant (182).png,test
BUSI,malignant (183).png,train
BUSI,malignant (184).png,train
BUSI,malignant (185).png,train
BUSI,malignant (186).png,train
BUSI,malignant (187).png,train
BUSI,malignant (188).png,train
BUSI,malignant (189).png,test
BUSI,malignant (19).png,val
BUSI,malignant (190).png,train
BUSI,malignant (191).png,train
BUSI,malignant (192).png,test
BUSI,malignant (193).png,val
BUSI,malignant (194).png,train
BUSI,malignant (195).png,train
BUSI,malignant (196).png,test
BUSI,malignant (197).png,train
BUSI,malignant (198).png,train
BUSI,malignant (199).png,val
BUSI,malignant (2).png,train
BUSI,malignant (20).png,train
BUSI,malignant (200).png,train
BUSI,malignant (201).png,train
BUSI,malignant (202).png,train
BUSI,malignant (203).png,train
BUSI,malignant (204).png,test
BUSI,malignant (205).png,val
BUSI,malignant (206).png,train
BUSI,malignant (207).png,train
BUSI,malignant (208).png,train
BUSI,malignant (209).png,val
BUSI,malignant (21).png,train
BUSI,malignant (210).png,train
BUSI,malignant (22).png,train
BUSI,malignant (23).png,train
BUSI,malignant (24).png,test
BUSI,malignant (25).png,train
BUSI,malignant (26).png,val
BUSI,malignant (27).png,train
BUSI,malignant (28).png,train
BUSI,malignant (29).png,train
BUSI,malignant (3).png,train
BUSI,malignant (30).png,train
BUSI,malignant (31).png,train
BUSI,malignant (32).png,train
BUSI,malignant (33).png,train
BUSI,malignant (34).png,test
BUSI,malignant (35).png,train
BUSI,malignant (36).png,test
BUSI,malignant (37).png,train
BUSI,malignant (38).png,val
BUSI,malignant (39).png,train
BUSI,malignant (4).png,train
BUSI,malignant (40).png,train
BUSI,malignant (41).png,test
BUSI,malignant (42).png,val
BUSI,malignant (43).png,train
BUSI,malignant (44).png,train
BUSI,malignant (45).png,train
BUSI,malignant (46).png,train
BUSI,malignant (47).png,train
BUSI,malignant (48).png,test
BUSI,malignant (49).png,val
BUSI,malignant (5).png,train
BUSI,malignant (50).png,train
BUSI,malignant (53).png,test
BUSI,malignant (54).png,train
BUSI,malignant (55).png,train
BUSI,malignant (56).png,val
BUSI,malignant (57).png,train
BUSI,malignant (58).png,test
BUSI,malignant (59).png,train
BUSI,malignant (6).png,train
BUSI,malignant (60).png,train
BUSI,malignant (61).png,train
BUSI,malignant (62).png,val
BUSI,malignant (63).png,train
BUSI,malignant (64).png,train
BUSI,malignant (65).png,train
BUSI,malignant (66).png,test
BUSI,malignant (67).png,train
BUSI,malignant (68).png,val
BUSI,malignant (69).png,train
BUSI,malignant (7).png,train
BUSI,malignant (70).png,train
BUSI,malignant (71).png,test
BUSI,malignant (72).png,train
BUSI,malignant (73).png,train
BUSI,malignant (74).png,train
BUSI,malignant (75).png,train
BUSI,malignant (76).png,train
BUSI,malignant (77).png,train
BUSI,malignant (78).png,train
BUSI,malignant (79).png,train
BUSI,malignant (8).png,val
BUSI,malignant (80).png,train
BUSI,malignant (81).png,test
BUSI,malignant (82).png,train
BUSI,malignant (83).png,val
BUSI,malignant (84).png,train
BUSI,malignant (85).png,train
BUSI,malignant (86).png,train
BUSI,malignant (87).png,train
BUSI,malignant (88).png,test
BUSI,malignant (89).png,test
BUSI,malignant (9).png,train
BUSI,malignant (90).png,train
BUSI,malignant (91).png,val
BUSI,malignant (92).png,train
BUSI,malignant (94).png,train
BUSI,malignant (95).png,train
BUSI,malignant (96).png,test
BUSI,malignant (97).png,val
BUSI,malignant (98).png,train
BUSI,malignant (99).png,train
BrEaST,case001.png,train
BrEaST,case002.png,train
BrEaST,case003.png,val
BrEaST,case004.png,train
BrEaST,case005.png,train
BrEaST,case006.png,train
BrEaST,case007.png,val
BrEaST,case008.png,test
BrEaST,case009.png,train
BrEaST,case010.png,train
BrEaST,case011.png,train
BrEaST,case012.png,train
BrEaST,case013.png,train
BrEaST,case014.png,val
BrEaST,case015.png,test
BrEaST,case016.png,train
BrEaST,case017.png,val
BrEaST,case018.png,train
BrEaST,case019.png,train
BrEaST,case020.png,train
BrEaST,case021.png,train
BrEaST,case022.png,test
BrEaST,case023.png,train
BrEaST,case024.png,train
BrEaST,case025.png,train
BrEaST,case026.png,train
BrEaST,case027.png,train
BrEaST,case028.png,train
BrEaST,case029.png,val
BrEaST,case030.png,train
BrEaST,case031.png,train
BrEaST,case032.png,train
BrEaST,case033.png,train
BrEaST,case034.png,test
BrEaST,case035.png,train
BrEaST,case036.png,val
BrEaST,case037.png,train
BrEaST,case038.png,train
BrEaST,case039.png,train
BrEaST,case040.png,train
BrEaST,case041.png,train
BrEaST,case042.png,train
BrEaST,case043.png,train
BrEaST,case044.png,train
BrEaST,case045.png,test
BrEaST,case046.png,val
BrEaST,case047.png,train
BrEaST,case048.png,train
BrEaST,case049.png,train
BrEaST,case050.png,train
BrEaST,case051.png,test
BrEaST,case052.png,train
BrEaST,case053.png,test
BrEaST,case054.png,train
BrEaST,case055.png,test
BrEaST,case056.png,train
BrEaST,case057.png,val
BrEaST,case058.png,train
BrEaST,case059.png,train
BrEaST,case060.png,test
BrEaST,case061.png,val
BrEaST,case062.png,train
BrEaST,case063.png,train
BrEaST,case064.png,val
BrEaST,case065.png,train
BrEaST,case066.png,train
BrEaST,case067.png,train
BrEaST,case068.png,train
BrEaST,case069.png,test
BrEaST,case070.png,train
BrEaST,case071.png,val
BrEaST,case072.png,train
BrEaST,case073.png,train
BrEaST,case074.png,train
BrEaST,case075.png,train
BrEaST,case076.png,test
BrEaST,case077.png,val
BrEaST,case078.png,train
BrEaST,case079.png,train
BrEaST,case080.png,train
BrEaST,case081.png,train
BrEaST,case082.png,train
BrEaST,case083.png,train
BrEaST,case084.png,val
BrEaST,case085.png,train
BrEaST,case086.png,test
BrEaST,case087.png,train
BrEaST,case088.png,train
BrEaST,case089.png,train
BrEaST,case090.png,train
BrEaST,case091.png,test
BrEaST,case092.png,train
BrEaST,case093.png,train
BrEaST,case094.png,val
BrEaST,case095.png,train
BrEaST,case096.png,train
BrEaST,case097.png,train
BrEaST,case098.png,train
BrEaST,case099.png,train
BrEaST,case100.png,train
BrEaST,case101.png,test
BrEaST,case102.png,test
BrEaST,case103.png,val
BrEaST,case104.png,val
BrEaST,case105.png,train
BrEaST,case106.png,train
BrEaST,case107.png,train
BrEaST,case108.png,train
BrEaST,case109.png,train
BrEaST,case110.png,train
BrEaST,case111.png,train
BrEaST,case112.png,test
BrEaST,case113.png,train
BrEaST,case114.png,train
BrEaST,case115.png,train
BrEaST,case116.png,train
BrEaST,case117.png,train
BrEaST,case118.png,train
BrEaST,case119.png,val
BrEaST,case120.png,train
BrEaST,case121.png,test
BrEaST,case122.png,train
BrEaST,case123.png,val
BrEaST,case124.png,train
BrEaST,case125.png,train
BrEaST,case126.png,train
BrEaST,case127.png,train
BrEaST,case128.png,train
BrEaST,case129.png,test
BrEaST,case130.png,val
BrEaST,case131.png,train
BrEaST,case132.png,train
BrEaST,case133.png,train
BrEaST,case134.png,train
BrEaST,case135.png,val
BrEaST,case136.png,train
BrEaST,case137.png,train
BrEaST,case138.png,train
BrEaST,case139.png,test
BrEaST,case140.png,train
BrEaST,case141.png,train
BrEaST,case142.png,train
BrEaST,case143.png,train
BrEaST,case144.png,train
BrEaST,case145.png,test
BrEaST,case146.png,train
BrEaST,case147.png,val
BrEaST,case148.png,train
BrEaST,case149.png,train
BrEaST,case150.png,train
BrEaST,case151.png,train
BrEaST,case152.png,test
BrEaST,case153.png,val
BrEaST,case154.png,train
BrEaST,case155.png,test
BrEaST,case156.png,train
BrEaST,case157.png,train
BrEaST,case158.png,val
BrEaST,case159.png,train
BrEaST,case160.png,train
BrEaST,case161.png,train
BrEaST,case162.png,train
BrEaST,case163.png,test
BrEaST,case164.png,train
BrEaST,case165.png,train
BrEaST,case166.png,val
BrEaST,case167.png,train
BrEaST,case168.png,train
BrEaST,case169.png,train
BrEaST,case170.png,train
BrEaST,case171.png,train
BrEaST,case172.png,test
BrEaST,case173.png,val
BrEaST,case174.png,test
BrEaST,case175.png,train
BrEaST,case176.png,train
BrEaST,case177.png,train
BrEaST,case178.png,train
BrEaST,case179.png,train
BrEaST,case180.png,train
BrEaST,case181.png,train
BrEaST,case182.png,test
BrEaST,case183.png,val
BrEaST,case184.png,train
BrEaST,case185.png,train
BrEaST,case186.png,train
BrEaST,case187.png,train
BrEaST,case188.png,train
BrEaST,case189.png,test
BrEaST,case190.png,train
BrEaST,case191.png,train
BrEaST,case192.png,val
BrEaST,case193.png,val
BrEaST,case194.png,train
BrEaST,case195.png,train
BrEaST,case196.png,test
BrEaST,case197.png,val
BrEaST,case198.png,train
BrEaST,case199.png,test
BrEaST,case200.png,train
BrEaST,case201.png,train
BrEaST,case202.png,train
BrEaST,case203.png,train
BrEaST,case204.png,train
BrEaST,case205.png,test
BrEaST,case206.png,train
BrEaST,case207.png,val
BrEaST,case208.png,train
BrEaST,case209.png,train
BrEaST,case210.png,train
BrEaST,case211.png,train
BrEaST,case212.png,val
BrEaST,case213.png,train
BrEaST,case214.png,train
BrEaST,case215.png,train
BrEaST,case216.png,test
BrEaST,case217.png,train
BrEaST,case218.png,val
BrEaST,case219.png,train
BrEaST,case220.png,train
BrEaST,case221.png,train
BrEaST,case222.png,train
BrEaST,case223.png,test
BrEaST,case224.png,train
BrEaST,case225.png,train
BrEaST,case226.png,train
BrEaST,case227.png,val
BrEaST,case228.png,train
BrEaST,case229.png,test
BrEaST,case230.png,train
BrEaST,case231.png,train
BrEaST,case232.png,train
BrEaST,case233.png,val
BrEaST,case234.png,train
BrEaST,case235.png,train
BrEaST,case236.png,val
BrEaST,case237.png,train
BrEaST,case238.png,train
BrEaST,case239.png,train
BrEaST,case240.png,train
BrEaST,case241.png,test
BrEaST,case242.png,val
BrEaST,case243.png,train
BrEaST,case244.png,train
BrEaST,case245.png,train
BrEaST,case246.png,train
BrEaST,case247.png,train
BrEaST,case248.png,test
BrEaST,case249.png,train
BrEaST,case250.png,train
BrEaST,case251.png,train
BrEaST,case252.png,train
BrEaST,case253.png,train
BrEaST,case254.png,val
BrEaST,case255.png,train
BrEaST,case256.png,train
BUS-BRA,bus_0001-l.png,test
BUS-BRA,bus_0001-r.png,test
BUS-BRA,bus_0002-l.png,test
BUS-BRA,bus_0002-r.png,test
BUS-BRA,bus_0003-l.png,train
BUS-BRA,bus_0003-r.png,train
BUS-BRA,bus_0004-l.png,test
BUS-BRA,bus_0004-r.png,test
BUS-BRA,bus_0005-l.png,val
BUS-BRA,bus_0005-r.png,val
BUS-BRA,bus_0006-s.png,test
BUS-BRA,bus_0007-l.png,val
BUS-BRA,bus_0007-r.png,val
BUS-BRA,bus_0008-l.png,train
BUS-BRA,bus_0008-r.png,train
BUS-BRA,bus_0009-l.png,train
BUS-BRA,bus_0009-r.png,train
BUS-BRA,bus_0010-l.png,train
BUS-BRA,bus_0010-r.png,train
BUS-BRA,bus_0011-l.png,test
BUS-BRA,bus_0011-r.png,test
BUS-BRA,bus_0012-l.png,train
BUS-BRA,bus_0012-r.png,train
BUS-BRA,bus_0013-l.png,train
BUS-BRA,bus_0013-r.png,train
BUS-BRA,bus_0014-s.png,test
BUS-BRA,bus_0015-s.png,test
BUS-BRA,bus_0016-l.png,val
BUS-BRA,bus_0016-r.png,val
BUS-BRA,bus_0017-s.png,train
BUS-BRA,bus_0018-s.png,test
BUS-BRA,bus_0019-l.png,train
BUS-BRA,bus_0019-r.png,train
BUS-BRA,bus_0020-l.png,train
BUS-BRA,bus_0020-r.png,train
BUS-BRA,bus_0021-l.png,train
BUS-BRA,bus_0021-r.png,train
BUS-BRA,bus_0022-l.png,train
BUS-BRA,bus_0022-r.png,train
BUS-BRA,bus_0023-l.png,train
BUS-BRA,bus_0023-r.png,train
BUS-BRA,bus_0024-l.png,train
BUS-BRA,bus_0024-r.png,train
BUS-BRA,bus_0025-l.png,val
BUS-BRA,bus_0025-r.png,val
BUS-BRA,bus_0026-s.png,train
BUS-BRA,bus_0027-l.png,train
BUS-BRA,bus_0027-r.png,train
BUS-BRA,bus_0028-l.png,train
BUS-BRA,bus_0028-r.png,train
BUS-BRA,bus_0029-s.png,test
BUS-BRA,bus_0030-s.png,train
BUS-BRA,bus_0031-s.png,test
BUS-BRA,bus_0032-l.png,test
BUS-BRA,bus_0032-r.png,test
BUS-BRA,bus_0033-l.png,val
BUS-BRA,bus_0033-r.png,val
BUS-BRA,bus_0034-l.png,train
BUS-BRA,bus_0034-r.png,train
BUS-BRA,bus_0035-l.png,train
BUS-BRA,bus_0035-r.png,train
BUS-BRA,bus_0036-l.png,train
BUS-BRA,bus_0036-r.png,train
BUS-BRA,bus_0037-s.png,train
BUS-BRA,bus_0038-s.png,train
BUS-BRA,bus_0039-l.png,train
BUS-BRA,bus_0039-r.png,train
BUS-BRA,bus_0040-s.png,train
BUS-BRA,bus_0041-l.png,train
BUS-BRA,bus_0041-r.png,train
BUS-BRA,bus_0042-l.png,train
BUS-BRA,bus_0042-r.png,train
BUS-BRA,bus_0043-s.png,test
BUS-BRA,bus_0044-l.png,train
BUS-BRA,bus_0044-r.png,train
BUS-BRA,bus_0045-l.png,test
BUS-BRA,bus_0045-r.png,test
BUS-BRA,bus_0046-l.png,val
BUS-BRA,bus_0046-r.png,val
BUS-BRA,bus_0047-l.png,train
BUS-BRA,bus_0047-r.png,train
BUS-BRA,bus_0048-l.png,test
BUS-BRA,bus_0048-r.png,test
BUS-BRA,bus_0049-l.png,train
BUS-BRA,bus_0049-r.png,train
BUS-BRA,bus_0050-l.png,train
BUS-BRA,bus_0050-r.png,train
BUS-BRA,bus_0051-l.png,train
BUS-BRA,bus_0051-r.png,train
BUS-BRA,bus_0052-l.png,train
BUS-BRA,bus_0052-r.png,train
BUS-BRA,bus_0053-l.png,test
BUS-BRA,bus_0053-r.png,test
BUS-BRA,bus_0054-s.png,train
BUS-BRA,bus_0055-l.png,train
BUS-BRA,bus_0055-r.png,train
BUS-BRA,bus_0056-l.png,test
BUS-BRA,bus_0056-r.png,test
BUS-BRA,bus_0057-l.png,test
BUS-BRA,bus_0057-r.png,test
BUS-BRA,bus_0058-l.png,test
BUS-BRA,bus_0058-r.png,test
BUS-BRA,bus_0059-l.png,train
BUS-BRA,bus_0059-r.png,train
BUS-BRA,bus_0060-l.png,test
BUS-BRA,bus_0060-r.png,test
BUS-BRA,bus_0061-l.png,val
BUS-BRA,bus_0061-r.png,val
BUS-BRA,bus_0062-l.png,train
BUS-BRA,bus_0062-r.png,train
BUS-BRA,bus_0063-s.png,train
BUS-BRA,bus_0064-s.png,train
BUS-BRA,bus_0065-l.png,train
BUS-BRA,bus_0065-r.png,train
BUS-BRA,bus_0066-l.png,train
BUS-BRA,bus_0066-r.png,train
BUS-BRA,bus_0067-l.png,train
BUS-BRA,bus_0067-r.png,train
BUS-BRA,bus_0068-l.png,train
BUS-BRA,bus_0068-r.png,train
BUS-BRA,bus_0069-l.png,train
BUS-BRA,bus_0069-r.png,train
BUS-BRA,bus_0070-l.png,test
BUS-BRA,bus_0070-r.png,test
BUS-BRA,bus_0071-l.png,test
BUS-BRA,bus_0071-r.png,test
BUS-BRA,bus_0072-l.png,train
BUS-BRA,bus_0072-r.png,train
BUS-BRA,bus_0073-l.png,test
BUS-BRA,bus_0073-r.png,test
BUS-BRA,bus_0074-s.png,train
BUS-BRA,bus_0075-l.png,test
BUS-BRA,bus_0075-r.png,test
BUS-BRA,bus_0076-l.png,test
BUS-BRA,bus_0076-r.png,test
BUS-BRA,bus_0077-s.png,val
BUS-BRA,bus_0078-s.png,train
BUS-BRA,bus_0079-l.png,test
BUS-BRA,bus_0079-r.png,test
BUS-BRA,bus_0080-l.png,train
BUS-BRA,bus_0080-r.png,train
BUS-BRA,bus_0081-s.png,train
BUS-BRA,bus_0082-s.png,val
BUS-BRA,bus_0083-l.png,train
BUS-BRA,bus_0083-r.png,train
BUS-BRA,bus_0084-l.png,train
BUS-BRA,bus_0084-r.png,train
BUS-BRA,bus_0085-l.png,test
BUS-BRA,bus_0085-r.png,test
BUS-BRA,bus_0086-l.png,train
BUS-BRA,bus_0086-r.png,train
BUS-BRA,bus_0087-l.png,test
BUS-BRA,bus_0087-r.png,test
BUS-BRA,bus_0088-s.png,train
BUS-BRA,bus_0089-s.png,train
BUS-BRA,bus_0090-l.png,train
BUS-BRA,bus_0090-r.png,train
BUS-BRA,bus_0091-l.png,test
BUS-BRA,bus_0091-r.png,test
BUS-BRA,bus_0092-l.png,val
BUS-BRA,bus_0092-r.png,val
BUS-BRA,bus_0093-s.png,train
BUS-BRA,bus_0094-l.png,train
BUS-BRA,bus_0094-r.png,train
BUS-BRA,bus_0095-s.png,train
BUS-BRA,bus_0096-s.png,train
BUS-BRA,bus_0097-l.png,train
BUS-BRA,bus_0097-r.png,train
BUS-BRA,bus_0098-l.png,train
BUS-BRA,bus_0098-r.png,train
BUS-BRA,bus_0099-l.png,test
BUS-BRA,bus_0099-r.png,test
BUS-BRA,bus_0100-l.png,test
BUS-BRA,bus_0100-r.png,test
BUS-BRA,bus_0101-l.png,test
BUS-BRA,bus_0101-r.png,test
BUS-BRA,bus_0102-l.png,train
BUS-BRA,bus_0102-r.png,train
BUS-BRA,bus_0103-s.png,train
BUS-BRA,bus_0104-s.png,val
BUS-BRA,bus_0105-l.png,train
BUS-BRA,bus_0105-r.png,train
BUS-BRA,bus_0106-l.png,train
BUS-BRA,bus_0106-r.png,train
BUS-BRA,bus_0107-l.png,train
BUS-BRA,bus_0107-r.png,train
BUS-BRA,bus_0108-l.png,train
BUS-BRA,bus_0108-r.png,train
BUS-BRA,bus_0109-s.png,test
BUS-BRA,bus_0110-l.png,train
BUS-BRA,bus_0110-r.png,train
BUS-BRA,bus_0111-l.png,test
BUS-BRA,bus_0111-r.png,test
BUS-BRA,bus_0112-l.png,train
BUS-BRA,bus_0112-r.png,train
BUS-BRA,bus_0113-s.png,train
BUS-BRA,bus_0114-s.png,test
BUS-BRA,bus_0115-l.png,val
BUS-BRA,bus_0115-r.png,val
BUS-BRA,bus_0116-l.png,train
BUS-BRA,bus_0116-r.png,train
BUS-BRA,bus_0117-l.png,train
BUS-BRA,bus_0117-r.png,train
BUS-BRA,bus_0118-l.png,train
BUS-BRA,bus_0118-r.png,train
BUS-BRA,bus_0119-l.png,train
BUS-BRA,bus_0119-r.png,train
BUS-BRA,bus_0120-l.png,train
BUS-BRA,bus_0120-r.png,train
BUS-BRA,bus_0121-s.png,test
BUS-BRA,bus_0122-s.png,test
BUS-BRA,bus_0123-l.png,test
BUS-BRA,bus_0123-r.png,test
BUS-BRA,bus_0124-l.png,train
BUS-BRA,bus_0124-r.png,train
BUS-BRA,bus_0125-l.png,train
BUS-BRA,bus_0125-r.png,train
BUS-BRA,bus_0126-s.png,train
BUS-BRA,bus_0127-s.png,val
BUS-BRA,bus_0128-s.png,train
BUS-BRA,bus_0129-l.png,train
BUS-BRA,bus_0129-r.png,train
BUS-BRA,bus_0130-l.png,train
BUS-BRA,bus_0130-r.png,train
BUS-BRA,bus_0131-l.png,test
BUS-BRA,bus_0131-r.png,test
BUS-BRA,bus_0132-l.png,test
BUS-BRA,bus_0132-r.png,test
BUS-BRA,bus_0133-l.png,test
BUS-BRA,bus_0133-r.png,test
BUS-BRA,bus_0134-l.png,test
BUS-BRA,bus_0134-r.png,test
BUS-BRA,bus_0135-l.png,test
BUS-BRA,bus_0135-r.png,test
BUS-BRA,bus_0136-l.png,test
BUS-BRA,bus_0136-r.png,test
BUS-BRA,bus_0137-l.png,train
BUS-BRA,bus_0137-r.png,train
BUS-BRA,bus_0138-s.png,val
BUS-BRA,bus_0139-l.png,val
BUS-BRA,bus_0139-r.png,val
BUS-BRA,bus_0140-l.png,train
BUS-BRA,bus_0140-r.png,train
BUS-BRA,bus_0141-l.png,train
BUS-BRA,bus_0141-r.png,train
BUS-BRA,bus_0142-l.png,train
BUS-BRA,bus_0142-r.png,train
BUS-BRA,bus_0143-l.png,train
BUS-BRA,bus_0143-r.png,train
BUS-BRA,bus_0144-l.png,train
BUS-BRA,bus_0144-r.png,train
BUS-BRA,bus_0145-l.png,train
BUS-BRA,bus_0145-r.png,train
BUS-BRA,bus_0146-l.png,train
BUS-BRA,bus_0146-r.png,train
BUS-BRA,bus_0147-l.png,train
BUS-BRA,bus_0147-r.png,train
BUS-BRA,bus_0148-l.png,train
BUS-BRA,bus_0148-r.png,train
BUS-BRA,bus_0149-l.png,test
BUS-BRA,bus_0149-r.png,test
BUS-BRA,bus_0150-l.png,test
BUS-BRA,bus_0150-r.png,test
BUS-BRA,bus_0151-l.png,val
BUS-BRA,bus_0151-r.png,val
BUS-BRA,bus_0152-l.png,test
BUS-BRA,bus_0152-r.png,test
BUS-BRA,bus_0153-l.png,train
BUS-BRA,bus_0153-r.png,train
BUS-BRA,bus_0154-l.png,train
BUS-BRA,bus_0154-r.png,train
BUS-BRA,bus_0155-l.png,test
BUS-BRA,bus_0155-r.png,test
BUS-BRA,bus_0156-l.png,train
BUS-BRA,bus_0156-r.png,train
BUS-BRA,bus_0157-l.png,train
BUS-BRA,bus_0157-r.png,train
BUS-BRA,bus_0158-l.png,train
BUS-BRA,bus_0158-r.png,train
BUS-BRA,bus_0159-l.png,train
BUS-BRA,bus_0159-r.png,train
BUS-BRA,bus_0160-l.png,train
BUS-BRA,bus_0160-r.png,train
BUS-BRA,bus_0161-l.png,train
BUS-BRA,bus_0161-r.png,train
BUS-BRA,bus_0162-l.png,test
BUS-BRA,bus_0162-r.png,test
BUS-BRA,bus_0163-l.png,test
BUS-BRA,bus_0163-r.png,test
BUS-BRA,bus_0164-l.png,train
BUS-BRA,bus_0164-r.png,train
BUS-BRA,bus_0165-l.png,test
BUS-BRA,bus_0165-r.png,test
BUS-BRA,bus_0166-l.png,val
BUS-BRA,bus_0166-r.png,val
BUS-BRA,bus_0167-l.png,train
BUS-BRA,bus_0167-r.png,train
BUS-BRA,bus_0168-l.png,train
BUS-BRA,bus_0168-r.png,train
BUS-BRA,bus_0169-l.png,train
BUS-BRA,bus_0169-r.png,train
BUS-BRA,bus_0170-l.png,train
BUS-BRA,bus_0170-r.png,train
BUS-BRA,bus_0171-l.png,train
BUS-BRA,bus_0171-r.png,train
BUS-BRA,bus_0172-l.png,train
BUS-BRA,bus_0172-r.png,train
BUS-BRA,bus_0173-l.png,train
BUS-BRA,bus_0173-r.png,train
BUS-BRA,bus_0174-l.png,test
BUS-BRA,bus_0174-r.png,test
BUS-BRA,bus_0175-l.png,test
BUS-BRA,bus_0175-r.png,test
BUS-BRA,bus_0176-l.png,train
BUS-BRA,bus_0176-r.png,train
BUS-BRA,bus_0177-l.png,test
BUS-BRA,bus_0177-r.png,test
BUS-BRA,bus_0178-l.png,test
BUS-BRA,bus_0178-r.png,test
BUS-BRA,bus_0179-l.png,val
BUS-BRA,bus_0179-r.png,val
BUS-BRA,bus_0180-l.png,train
BUS-BRA,bus_0180-r.png,train
BUS-BRA,bus_0181-s.png,train
BUS-BRA,bus_0182-l.png,train
BUS-BRA,bus_0182-r.png,train
BUS-BRA,bus_0183-l.png,train
BUS-BRA,bus_0183-r.png,train
BUS-BRA,bus_0184-l.png,train
BUS-BRA,bus_0184-r.png,train
BUS-BRA,bus_0185-l.png,train
BUS-BRA,bus_0185-r.png,train
BUS-BRA,bus_0186-l.png,val
BUS-BRA,bus_0186-r.png,val
BUS-BRA,bus_0187-l.png,test
BUS-BRA,bus_0187-r.png,test
BUS-BRA,bus_0188-l.png,test
BUS-BRA,bus_0188-r.png,test
BUS-BRA,bus_0189-l.png,test
BUS-BRA,bus_0189-r.png,test
BUS-BRA,bus_0190-l.png,train
BUS-BRA,bus_0190-r.png,train
BUS-BRA,bus_0191-l.png,train
BUS-BRA,bus_0191-r.png,train
BUS-BRA,bus_0192-l.png,train
BUS-BRA,bus_0192-r.png,train
BUS-BRA,bus_0193-l.png,train
BUS-BRA,bus_0193-r.png,train
BUS-BRA,bus_0194-l.png,train
BUS-BRA,bus_0194-r.png,train
BUS-BRA,bus_0195-l.png,train
BUS-BRA,bus_0195-r.png,train
BUS-BRA,bus_0196-l.png,test
BUS-BRA,bus_0196-r.png,test
BUS-BRA,bus_0197-l.png,train
BUS-BRA,bus_0197-r.png,train
BUS-BRA,bus_0198-l.png,test
BUS-BRA,bus_0198-r.png,test
BUS-BRA,bus_0199-l.png,train
BUS-BRA,bus_0199-r.png,train
BUS-BRA,bus_0200-s.png,train
BUS-BRA,bus_0201-s.png,test
BUS-BRA,bus_0202-l.png,test
BUS-BRA,bus_0202-r.png,test
BUS-BRA,bus_0203-l.png,train
BUS-BRA,bus_0203-r.png,train
BUS-BRA,bus_0204-l.png,train
BUS-BRA,bus_0204-r.png,train
BUS-BRA,bus_0205-l.png,val
BUS-BRA,bus_0205-r.png,val
BUS-BRA,bus_0206-l.png,train
BUS-BRA,bus_0206-r.png,train
BUS-BRA,bus_0207-l.png,train
BUS-BRA,bus_0207-r.png,train
BUS-BRA,bus_0208-l.png,train
BUS-BRA,bus_0208-r.png,train
BUS-BRA,bus_0209-l.png,test
BUS-BRA,bus_0209-r.png,test
BUS-BRA,bus_0210-l.png,test
BUS-BRA,bus_0210-r.png,test
BUS-BRA,bus_0211-l.png,val
BUS-BRA,bus_0211-r.png,val
BUS-BRA,bus_0212-l.png,test
BUS-BRA,bus_0212-r.png,test
BUS-BRA,bus_0213-l.png,test
BUS-BRA,bus_0213-r.png,test
BUS-BRA,bus_0214-l.png,train
BUS-BRA,bus_0214-r.png,train
BUS-BRA,bus_0215-l.png,train
BUS-BRA,bus_0215-r.png,train
BUS-BRA,bus_0216-l.png,train
BUS-BRA,bus_0216-r.png,train
BUS-BRA,bus_0217-l.png,val
BUS-BRA,bus_0217-r.png,val
BUS-BRA,bus_0218-l.png,test
BUS-BRA,bus_0218-r.png,test
BUS-BRA,bus_0219-l.png,train
BUS-BRA,bus_0219-r.png,train
BUS-BRA,bus_0220-l.png,train
BUS-BRA,bus_0220-r.png,train
BUS-BRA,bus_0221-l.png,train
BUS-BRA,bus_0221-r.png,train
BUS-BRA,bus_0222-l.png,test
BUS-BRA,bus_0222-r.png,test
BUS-BRA,bus_0223-l.png,train
BUS-BRA,bus_0223-r.png,train
BUS-BRA,bus_0224-l.png,test
BUS-BRA,bus_0224-r.png,test
BUS-BRA,bus_0225-l.png,test
BUS-BRA,bus_0225-r.png,test
BUS-BRA,bus_0226-l.png,val
BUS-BRA,bus_0226-r.png,val
BUS-BRA,bus_0227-l.png,test
BUS-BRA,bus_0227-r.png,test
BUS-BRA,bus_0228-l.png,train
BUS-BRA,bus_0228-r.png,train
BUS-BRA,bus_0229-l.png,train
BUS-BRA,bus_0229-r.png,train
BUS-BRA,bus_0230-l.png,train
BUS-BRA,bus_0230-r.png,train
BUS-BRA,bus_0231-l.png,train
BUS-BRA,bus_0231-r.png,train
BUS-BRA,bus_0232-l.png,train
BUS-BRA,bus_0232-r.png,train
BUS-BRA,bus_0233-s.png,train
BUS-BRA,bus_0234-l.png,test
BUS-BRA,bus_0234-r.png,test
BUS-BRA,bus_0235-l.png,test
BUS-BRA,bus_0235-r.png,test
BUS-BRA,bus_0236-l.png,test
BUS-BRA,bus_0236-r.png,test
BUS-BRA,bus_0237-l.png,val
BUS-BRA,bus_0237-r.png,val
BUS-BRA,bus_0238-l.png,train
BUS-BRA,bus_0238-r.png,train
BUS-BRA,bus_0239-l.png,train
BUS-BRA,bus_0239-r.png,train
BUS-BRA,bus_0240-l.png,train
BUS-BRA,bus_0240-r.png,train
BUS-BRA,bus_0241-l.png,train
BUS-BRA,bus_0241-r.png,train
BUS-BRA,bus_0242-l.png,train
BUS-BRA,bus_0242-r.png,train
BUS-BRA,bus_0243-l.png,train
BUS-BRA,bus_0243-r.png,train
BUS-BRA,bus_0244-l.png,train
BUS-BRA,bus_0244-r.png,train
BUS-BRA,bus_0245-l.png,train
BUS-BRA,bus_0245-r.png,train
BUS-BRA,bus_0246-l.png,train
BUS-BRA,bus_0246-r.png,train
BUS-BRA,bus_0247-l.png,val
BUS-BRA,bus_0247-r.png,val
BUS-BRA,bus_0248-l.png,train
BUS-BRA,bus_0248-r.png,train
BUS-BRA,bus_0249-l.png,test
BUS-BRA,bus_0249-r.png,test
BUS-BRA,bus_0250-l.png,test
BUS-BRA,bus_0250-r.png,test
BUS-BRA,bus_0251-l.png,train
BUS-BRA,bus_0251-r.png,train
BUS-BRA,bus_0252-l.png,test
BUS-BRA,bus_0252-r.png,test
BUS-BRA,bus_0253-l.png,train
BUS-BRA,bus_0253-r.png,train
BUS-BRA,bus_0254-l.png,train
BUS-BRA,bus_0254-r.png,train
BUS-BRA,bus_0255-l.png,val
BUS-BRA,bus_0255-r.png,val
BUS-BRA,bus_0256-l.png,train
BUS-BRA,bus_0256-r.png,train
BUS-BRA,bus_0257-l.png,train
BUS-BRA,bus_0257-r.png,train
BUS-BRA,bus_0258-l.png,train
BUS-BRA,bus_0258-r.png,train
BUS-BRA,bus_0259-l.png,train
BUS-BRA,bus_0259-r.png,train
BUS-BRA,bus_0260-l.png,test
BUS-BRA,bus_0260-r.png,test
BUS-BRA,bus_0261-l.png,test
BUS-BRA,bus_0261-r.png,test
BUS-BRA,bus_0262-l.png,train
BUS-BRA,bus_0262-r.png,train
BUS-BRA,bus_0263-l.png,test
BUS-BRA,bus_0263-r.png,test
BUS-BRA,bus_0264-l.png,val
BUS-BRA,bus_0264-r.png,val
BUS-BRA,bus_0265-s.png,train
BUS-BRA,bus_0266-l.png,train
BUS-BRA,bus_0266-r.png,train
BUS-BRA,bus_0267-l.png,train
BUS-BRA,bus_0267-r.png,train
BUS-BRA,bus_0268-l.png,train
BUS-BRA,bus_0268-r.png,train
BUS-BRA,bus_0269-l.png,test
BUS-BRA,bus_0269-r.png,test
BUS-BRA,bus_0270-l.png,test
BUS-BRA,bus_0270-r.png,test
BUS-BRA,bus_0271-l.png,train
BUS-BRA,bus_0271-r.png,train
BUS-BRA,bus_0272-l.png,test
BUS-BRA,bus_0272-r.png,test
BUS-BRA,bus_0273-l.png,train
BUS-BRA,bus_0273-r.png,train
BUS-BRA,bus_0274-l.png,val
BUS-BRA,bus_0274-r.png,val
BUS-BRA,bus_0275-l.png,test
BUS-BRA,bus_0275-r.png,test
BUS-BRA,bus_0276-l.png,train
BUS-BRA,bus_0276-r.png,train
BUS-BRA,bus_0277-l.png,test
BUS-BRA,bus_0277-r.png,test
BUS-BRA,bus_0278-s.png,test
BUS-BRA,bus_0279-l.png,val
BUS-BRA,bus_0279-r.png,val
BUS-BRA,bus_0280-l.png,train
BUS-BRA,bus_0280-r.png,train
BUS-BRA,bus_0281-l.png,train
BUS-BRA,bus_0281-r.png,train
BUS-BRA,bus_0282-l.png,train
BUS-BRA,bus_0282-r.png,train
BUS-BRA,bus_0283-l.png,train
BUS-BRA,bus_0283-r.png,train
BUS-BRA,bus_0284-l.png,train
BUS-BRA,bus_0284-r.png,train
BUS-BRA,bus_0285-l.png,train
BUS-BRA,bus_0285-r.png,train
BUS-BRA,bus_0286-l.png,train
BUS-BRA,bus_0286-r.png,train
BUS-BRA,bus_0287-l.png,test
BUS-BRA,bus_0287-r.png,test
BUS-BRA,bus_0288-l.png,train
BUS-BRA,bus_0288-r.png,train
BUS-BRA,bus_0289-l.png,train
BUS-BRA,bus_0289-r.png,train
BUS-BRA,bus_0290-l.png,test
BUS-BRA,bus_0290-r.png,test
BUS-BRA,bus_0291-l.png,test
BUS-BRA,bus_0291-r.png,test
BUS-BRA,bus_0292-l.png,val
BUS-BRA,bus_0292-r.png,val
BUS-BRA,bus_0293-s.png,train
BUS-BRA,bus_0294-l.png,train
BUS-BRA,bus_0294-r.png,train
BUS-BRA,bus_0295-l.png,train
BUS-BRA,bus_0295-r.png,train
BUS-BRA,bus_0296-l.png,train
BUS-BRA,bus_0296-r.png,train
BUS-BRA,bus_0297-l.png,train
BUS-BRA,bus_0297-r.png,train
BUS-BRA,bus_0298-l.png,test
BUS-BRA,bus_0298-r.png,test
BUS-BRA,bus_0299-l.png,test
BUS-BRA,bus_0299-r.png,test
BUS-BRA,bus_0300-l.png,train
BUS-BRA,bus_0300-r.png,train
BUS-BRA,bus_0301-l.png,train
BUS-BRA,bus_0301-r.png,train
BUS-BRA,bus_0302-s.png,train
BUS-BRA,bus_0303-l.png,test
BUS-BRA,bus_0303-r.png,test
BUS-BRA,bus_0304-l.png,train
BUS-BRA,bus_0304-r.png,train
BUS-BRA,bus_0305-l.png,val
BUS-BRA,bus_0305-r.png,val
BUS-BRA,bus_0306-l.png,train
BUS-BRA,bus_0306-r.png,train
BUS-BRA,bus_0307-l.png,train
BUS-BRA,bus_0307-r.png,train
BUS-BRA,bus_0308-l.png,train
BUS-BRA,bus_0308-r.png,train
BUS-BRA,bus_0309-l.png,train
BUS-BRA,bus_0309-r.png,train
BUS-BRA,bus_0310-l.png,test
BUS-BRA,bus_0310-r.png,test
BUS-BRA,bus_0311-l.png,train
BUS-BRA,bus_0311-r.png,train
BUS-BRA,bus_0312-s.png,test
BUS-BRA,bus_0313-l.png,test
BUS-BRA,bus_0313-r.png,test
BUS-BRA,bus_0314-l.png,test
BUS-BRA,bus_0314-r.png,test
BUS-BRA,bus_0315-l.png,train
BUS-BRA,bus_0315-r.png,train
BUS-BRA,bus_0316-l.png,val
BUS-BRA,bus_0316-r.png,val
BUS-BRA,bus_0317-l.png,train
BUS-BRA,bus_0317-r.png,train
BUS-BRA,bus_0318-l.png,train
BUS-BRA,bus_0318-r.png,train
BUS-BRA,bus_0319-l.png,train
BUS-BRA,bus_0319-r.png,train
BUS-BRA,bus_0320-l.png,test
BUS-BRA,bus_0320-r.png,test
BUS-BRA,bus_0321-s.png,test
BUS-BRA,bus_0322-s.png,train
BUS-BRA,bus_0323-s.png,train
BUS-BRA,bus_0324-s.png,test
BUS-BRA,bus_0325-l.png,test
BUS-BRA,bus_0325-r.png,test
BUS-BRA,bus_0326-l.png,val
BUS-BRA,bus_0326-r.png,val
BUS-BRA,bus_0327-l.png,train
BUS-BRA,bus_0327-r.png,train
BUS-BRA,bus_0328-l.png,test
BUS-BRA,bus_0328-r.png,test
BUS-BRA,bus_0329-l.png,train
BUS-BRA,bus_0329-r.png,train
BUS-BRA,bus_0330-l.png,val
BUS-BRA,bus_0330-r.png,val
BUS-BRA,bus_0331-s.png,train
BUS-BRA,bus_0332-l.png,train
BUS-BRA,bus_0332-r.png,train
BUS-BRA,bus_0333-s.png,train
BUS-BRA,bus_0334-l.png,train
BUS-BRA,bus_0334-r.png,train
BUS-BRA,bus_0335-l.png,train
BUS-BRA,bus_0335-r.png,train
BUS-BRA,bus_0336-s.png,train
BUS-BRA,bus_0337-l.png,train
BUS-BRA,bus_0337-r.png,train
BUS-BRA,bus_0338-l.png,train
BUS-BRA,bus_0338-r.png,train
BUS-BRA,bus_0339-l.png,test
BUS-BRA,bus_0339-r.png,test
BUS-BRA,bus_0340-l.png,test
BUS-BRA,bus_0340-r.png,test
BUS-BRA,bus_0341-l.png,test
BUS-BRA,bus_0341-r.png,test
BUS-BRA,bus_0342-l.png,val
BUS-BRA,bus_0342-r.png,val
BUS-BRA,bus_0343-l.png,train
BUS-BRA,bus_0343-r.png,train
BUS-BRA,bus_0344-l.png,train
BUS-BRA,bus_0344-r.png,train
BUS-BRA,bus_0345-l.png,train
BUS-BRA,bus_0345-r.png,train
BUS-BRA,bus_0346-l.png,train
BUS-BRA,bus_0346-r.png,train
BUS-BRA,bus_0347-s.png,train
BUS-BRA,bus_0348-l.png,test
BUS-BRA,bus_0348-r.png,test
BUS-BRA,bus_0349-l.png,train
BUS-BRA,bus_0349-r.png,train
BUS-BRA,bus_0350-l.png,test
BUS-BRA,bus_0350-r.png,test
BUS-BRA,bus_0351-l.png,val
BUS-BRA,bus_0351-r.png,val
BUS-BRA,bus_0352-l.png,test
BUS-BRA,bus_0352-r.png,test
BUS-BRA,bus_0353-l.png,test
BUS-BRA,bus_0353-r.png,test
BUS-BRA,bus_0354-s.png,train
BUS-BRA,bus_0355-s.png,train
BUS-BRA,bus_0356-l.png,train
BUS-BRA,bus_0356-r.png,train
BUS-BRA,bus_0357-l.png,train
BUS-BRA,bus_0357-r.png,train
BUS-BRA,bus_0358-l.png,train
BUS-BRA,bus_0358-r.png,train
BUS-BRA,bus_0359-l.png,train
BUS-BRA,bus_0359-r.png,train
BUS-BRA,bus_0360-l.png,test
BUS-BRA,bus_0360-r.png,test
BUS-BRA,bus_0361-l.png,test
BUS-BRA,bus_0361-r.png,test
BUS-BRA,bus_0362-l.png,test
BUS-BRA,bus_0362-r.png,test
BUS-BRA,bus_0363-l.png,test
BUS-BRA,bus_0363-r.png,test
BUS-BRA,bus_0364-l.png,test
BUS-BRA,bus_0364-r.png,test
BUS-BRA,bus_0365-l.png,val
BUS-BRA,bus_0365-r.png,val
BUS-BRA,bus_0366-l.png,train
BUS-BRA,bus_0366-r.png,train
BUS-BRA,bus_0367-l.png,train
BUS-BRA,bus_0367-r.png,train
BUS-BRA,bus_0368-l.png,train
BUS-BRA,bus_0368-r.png,train
BUS-BRA,bus_0369-l.png,train
BUS-BRA,bus_0369-r.png,train
BUS-BRA,bus_0370-l.png,train
BUS-BRA,bus_0370-r.png,train
BUS-BRA,bus_0371-l.png,train
BUS-BRA,bus_0371-r.png,train
BUS-BRA,bus_0372-l.png,train
BUS-BRA,bus_0372-r.png,train
BUS-BRA,bus_0373-l.png,test
BUS-BRA,bus_0373-r.png,test
BUS-BRA,bus_0374-l.png,train
BUS-BRA,bus_0374-r.png,train
BUS-BRA,bus_0375-l.png,test
BUS-BRA,bus_0375-r.png,test
BUS-BRA,bus_0376-l.png,test
BUS-BRA,bus_0376-r.png,test
BUS-BRA,bus_0377-l.png,val
BUS-BRA,bus_0377-r.png,val
BUS-BRA,bus_0378-l.png,train
BUS-BRA,bus_0378-r.png,train
BUS-BRA,bus_0379-l.png,train
BUS-BRA,bus_0379-r.png,train
BUS-BRA,bus_0380-l.png,train
BUS-BRA,bus_0380-r.png,train
BUS-BRA,bus_0381-l.png,train
BUS-BRA,bus_0381-r.png,train
BUS-BRA,bus_0382-l.png,train
BUS-BRA,bus_0382-r.png,train
BUS-BRA,bus_0383-l.png,test
BUS-BRA,bus_0383-r.png,test
BUS-BRA,bus_0384-l.png,val
BUS-BRA,bus_0384-r.png,val
BUS-BRA,bus_0385-l.png,train
BUS-BRA,bus_0385-r.png,train
BUS-BRA,bus_0386-l.png,train
BUS-BRA,bus_0386-r.png,train
BUS-BRA,bus_0387-l.png,test
BUS-BRA,bus_0387-r.png,test
BUS-BRA,bus_0388-l.png,train
BUS-BRA,bus_0388-r.png,train
BUS-BRA,bus_0389-l.png,test
BUS-BRA,bus_0389-r.png,test
BUS-BRA,bus_0390-l.png,val
BUS-BRA,bus_0390-r.png,val
BUS-BRA,bus_0391-l.png,train
BUS-BRA,bus_0391-r.png,train
BUS-BRA,bus_0392-l.png,train
BUS-BRA,bus_0392-r.png,train
BUS-BRA,bus_0393-l.png,train
BUS-BRA,bus_0393-r.png,train
BUS-BRA,bus_0394-l.png,test
BUS-BRA,bus_0394-r.png,test
BUS-BRA,bus_0395-l.png,test
BUS-BRA,bus_0395-r.png,test
BUS-BRA,bus_0396-l.png,test
BUS-BRA,bus_0396-r.png,test
BUS-BRA,bus_0397-l.png,val
BUS-BRA,bus_0397-r.png,val
BUS-BRA,bus_0398-l.png,train
BUS-BRA,bus_0398-r.png,train
BUS-BRA,bus_0399-l.png,train
BUS-BRA,bus_0399-r.png,train
BUS-BRA,bus_0400-l.png,train
BUS-BRA,bus_0400-r.png,train
BUS-BRA,bus_0401-l.png,train
BUS-BRA,bus_0401-r.png,train
BUS-BRA,bus_0402-l.png,train
BUS-BRA,bus_0402-r.png,train
BUS-BRA,bus_0403-l.png,train
BUS-BRA,bus_0403-r.png,train
BUS-BRA,bus_0404-l.png,train
BUS-BRA,bus_0404-r.png,train
BUS-BRA,bus_0405-l.png,train
BUS-BRA,bus_0405-r.png,train
BUS-BRA,bus_0406-l.png,test
BUS-BRA,bus_0406-r.png,test
BUS-BRA,bus_0407-l.png,train
BUS-BRA,bus_0407-r.png,train
BUS-BRA,bus_0408-l.png,train
BUS-BRA,bus_0408-r.png,train
BUS-BRA,bus_0409-l.png,test
BUS-BRA,bus_0409-r.png,test
BUS-BRA,bus_0410-l.png,test
BUS-BRA,bus_0410-r.png,test
BUS-BRA,bus_0411-l.png,train
BUS-BRA,bus_0411-r.png,train
BUS-BRA,bus_0412-l.png,val
BUS-BRA,bus_0412-r.png,val
BUS-BRA,bus_0413-l.png,train
BUS-BRA,bus_0413-r.png,train
BUS-BRA,bus_0414-l.png,train
BUS-BRA,bus_0414-r.png,train
BUS-BRA,bus_0415-l.png,train
BUS-BRA,bus_0415-r.png,train
BUS-BRA,bus_0416-l.png,train
BUS-BRA,bus_0416-r.png,train
BUS-BRA,bus_0417-l.png,train
BUS-BRA,bus_0417-r.png,train
BUS-BRA,bus_0418-l.png,test
BUS-BRA,bus_0418-r.png,test
BUS-BRA,bus_0419-l.png,test
BUS-BRA,bus_0419-r.png,test
BUS-BRA,bus_0420-l.png,test
BUS-BRA,bus_0420-r.png,test
BUS-BRA,bus_0421-l.png,train
BUS-BRA,bus_0421-r.png,train
BUS-BRA,bus_0422-l.png,train
BUS-BRA,bus_0422-r.png,train
BUS-BRA,bus_0423-l.png,train
BUS-BRA,bus_0423-r.png,train
BUS-BRA,bus_0424-l.png,train
BUS-BRA,bus_0424-r.png,train
BUS-BRA,bus_0425-l.png,test
BUS-BRA,bus_0425-r.png,test
BUS-BRA,bus_0426-l.png,val
BUS-BRA,bus_0426-r.png,val
BUS-BRA,bus_0427-l.png,test
BUS-BRA,bus_0427-r.png,test
BUS-BRA,bus_0428-l.png,train
BUS-BRA,bus_0428-r.png,train
BUS-BRA,bus_0429-l.png,test
BUS-BRA,bus_0429-r.png,test
BUS-BRA,bus_0430-l.png,train
BUS-BRA,bus_0430-r.png,train
BUS-BRA,bus_0431-l.png,train
BUS-BRA,bus_0431-r.png,train
BUS-BRA,bus_0432-l.png,val
BUS-BRA,bus_0432-r.png,val
BUS-BRA,bus_0433-l.png,train
BUS-BRA,bus_0433-r.png,train
BUS-BRA,bus_0434-l.png,test
BUS-BRA,bus_0434-r.png,test
BUS-BRA,bus_0435-l.png,train
BUS-BRA,bus_0435-r.png,train
BUS-BRA,bus_0436-l.png,test
BUS-BRA,bus_0436-r.png,test
BUS-BRA,bus_0437-l.png,train
BUS-BRA,bus_0437-r.png,train
BUS-BRA,bus_0438-l.png,test
BUS-BRA,bus_0438-r.png,test
BUS-BRA,bus_0439-l.png,train
BUS-BRA,bus_0439-r.png,train
BUS-BRA,bus_0440-l.png,val
BUS-BRA,bus_0440-r.png,val
BUS-BRA,bus_0441-l.png,train
BUS-BRA,bus_0441-r.png,train
BUS-BRA,bus_0442-l.png,train
BUS-BRA,bus_0442-r.png,train
BUS-BRA,bus_0443-l.png,train
BUS-BRA,bus_0443-r.png,train
BUS-BRA,bus_0444-l.png,val
BUS-BRA,bus_0444-r.png,val
BUS-BRA,bus_0445-l.png,train
BUS-BRA,bus_0445-r.png,train
BUS-BRA,bus_0446-l.png,train
BUS-BRA,bus_0446-r.png,train
BUS-BRA,bus_0447-l.png,train
BUS-BRA,bus_0447-r.png,train
BUS-BRA,bus_0448-l.png,test
BUS-BRA,bus_0448-r.png,test
BUS-BRA,bus_0449-l.png,test
BUS-BRA,bus_0449-r.png,test
BUS-BRA,bus_0450-l.png,train
BUS-BRA,bus_0450-r.png,train
BUS-BRA,bus_0451-l.png,test
BUS-BRA,bus_0451-r.png,test
BUS-BRA,bus_0452-l.png,train
BUS-BRA,bus_0452-r.png,train
BUS-BRA,bus_0453-l.png,train
BUS-BRA,bus_0453-r.png,train
BUS-BRA,bus_0454-l.png,train
BUS-BRA,bus_0454-r.png,train
BUS-BRA,bus_0455-l.png,train
BUS-BRA,bus_0455-r.png,train
BUS-BRA,bus_0456-l.png,val
BUS-BRA,bus_0456-r.png,val
BUS-BRA,bus_0457-l.png,train
BUS-BRA,bus_0457-r.png,train
BUS-BRA,bus_0458-l.png,train
BUS-BRA,bus_0458-r.png,train
BUS-BRA,bus_0459-l.png,test
BUS-BRA,bus_0459-r.png,test
BUS-BRA,bus_0460-l.png,test
BUS-BRA,bus_0460-r.png,test
BUS-BRA,bus_0461-l.png,test
BUS-BRA,bus_0461-r.png,test
BUS-BRA,bus_0462-l.png,train
BUS-BRA,bus_0462-r.png,train
BUS-BRA,bus_0463-l.png,train
BUS-BRA,bus_0463-r.png,train
BUS-BRA,bus_0464-l.png,train
BUS-BRA,bus_0464-r.png,train
BUS-BRA,bus_0465-l.png,train
BUS-BRA,bus_0465-r.png,train
BUS-BRA,bus_0466-l.png,test
BUS-BRA,bus_0466-r.png,test
BUS-BRA,bus_0467-l.png,train
BUS-BRA,bus_0467-r.png,train
BUS-BRA,bus_0468-l.png,test
BUS-BRA,bus_0468-r.png,test
BUS-BRA,bus_0469-l.png,train
BUS-BRA,bus_0469-r.png,train
BUS-BRA,bus_0470-l.png,test
BUS-BRA,bus_0470-r.png,test
BUS-BRA,bus_0471-l.png,train
BUS-BRA,bus_0471-r.png,train
BUS-BRA,bus_0472-l.png,train
BUS-BRA,bus_0472-r.png,train
BUS-BRA,bus_0473-l.png,val
BUS-BRA,bus_0473-r.png,val
BUS-BRA,bus_0474-l.png,train
BUS-BRA,bus_0474-r.png,train
BUS-BRA,bus_0475-l.png,train
BUS-BRA,bus_0475-r.png,train
BUS-BRA,bus_0476-l.png,train
BUS-BRA,bus_0476-r.png,train
BUS-BRA,bus_0477-l.png,train
BUS-BRA,bus_0477-r.png,train
BUS-BRA,bus_0478-l.png,train
BUS-BRA,bus_0478-r.png,train
BUS-BRA,bus_0479-l.png,train
BUS-BRA,bus_0479-r.png,train
BUS-BRA,bus_0480-l.png,val
BUS-BRA,bus_0480-r.png,val
BUS-BRA,bus_0481-l.png,test
BUS-BRA,bus_0481-r.png,test
BUS-BRA,bus_0482-l.png,train
BUS-BRA,bus_0482-r.png,train
BUS-BRA,bus_0483-l.png,train
BUS-BRA,bus_0483-r.png,train
BUS-BRA,bus_0484-l.png,test
BUS-BRA,bus_0484-r.png,test
BUS-BRA,bus_0485-l.png,test
BUS-BRA,bus_0485-r.png,test
BUS-BRA,bus_0486-l.png,test
BUS-BRA,bus_0486-r.png,test
BUS-BRA,bus_0487-l.png,train
BUS-BRA,bus_0487-r.png,train
BUS-BRA,bus_0488-l.png,train
BUS-BRA,bus_0488-r.png,train
BUS-BRA,bus_0489-l.png,test
BUS-BRA,bus_0489-r.png,test
BUS-BRA,bus_0490-l.png,train
BUS-BRA,bus_0490-r.png,train
BUS-BRA,bus_0491-l.png,train
BUS-BRA,bus_0491-r.png,train
BUS-BRA,bus_0492-l.png,train
BUS-BRA,bus_0492-r.png,train
BUS-BRA,bus_0493-l.png,train
BUS-BRA,bus_0493-r.png,train
BUS-BRA,bus_0494-l.png,train
BUS-BRA,bus_0494-r.png,train
BUS-BRA,bus_0495-l.png,test
BUS-BRA,bus_0495-r.png,test
BUS-BRA,bus_0496-l.png,train
BUS-BRA,bus_0496-r.png,train
BUS-BRA,bus_0497-l.png,test
BUS-BRA,bus_0497-r.png,test
BUS-BRA,bus_0498-l.png,train
BUS-BRA,bus_0498-r.png,train
BUS-BRA,bus_0499-l.png,test
BUS-BRA,bus_0499-r.png,test
BUS-BRA,bus_0500-l.png,val
BUS-BRA,bus_0500-r.png,val
BUS-BRA,bus_0501-l.png,train
BUS-BRA,bus_0501-r.png,train
BUS-BRA,bus_0502-l.png,train
BUS-BRA,bus_0502-r.png,train
BUS-BRA,bus_0503-l.png,test
BUS-BRA,bus_0503-r.png,test
BUS-BRA,bus_0504-l.png,val
BUS-BRA,bus_0504-r.png,val
BUS-BRA,bus_0505-l.png,train
BUS-BRA,bus_0505-r.png,train
BUS-BRA,bus_0506-l.png,test
BUS-BRA,bus_0506-r.png,test
BUS-BRA,bus_0507-l.png,test
BUS-BRA,bus_0507-r.png,test
BUS-BRA,bus_0508-l.png,test
BUS-BRA,bus_0508-r.png,test
BUS-BRA,bus_0509-l.png,train
BUS-BRA,bus_0509-r.png,train
BUS-BRA,bus_0510-l.png,val
BUS-BRA,bus_0510-r.png,val
BUS-BRA,bus_0511-l.png,train
BUS-BRA,bus_0511-r.png,train
BUS-BRA,bus_0512-l.png,train
BUS-BRA,bus_0512-r.png,train
BUS-BRA,bus_0513-l.png,train
BUS-BRA,bus_0513-r.png,train
BUS-BRA,bus_0514-l.png,train
BUS-BRA,bus_0514-r.png,train
BUS-BRA,bus_0515-l.png,train
BUS-BRA,bus_0515-r.png,train
BUS-BRA,bus_0516-l.png,train
BUS-BRA,bus_0516-r.png,train
BUS-BRA,bus_0517-l.png,train
BUS-BRA,bus_0517-r.png,train
BUS-BRA,bus_0518-l.png,train
BUS-BRA,bus_0518-r.png,train
BUS-BRA,bus_0519-l.png,test
BUS-BRA,bus_0519-r.png,test
BUS-BRA,bus_0520-l.png,train
BUS-BRA,bus_0520-r.png,train
BUS-BRA,bus_0521-l.png,test
BUS-BRA,bus_0521-r.png,test
BUS-BRA,bus_0522-l.png,test
BUS-BRA,bus_0522-r.png,test
BUS-BRA,bus_0523-l.png,val
BUS-BRA,bus_0523-r.png,val
BUS-BRA,bus_0524-l.png,train
BUS-BRA,bus_0524-r.png,train
BUS-BRA,bus_0525-l.png,train
BUS-BRA,bus_0525-r.png,train
BUS-BRA,bus_0526-l.png,train
BUS-BRA,bus_0526-r.png,train
BUS-BRA,bus_0527-l.png,train
BUS-BRA,bus_0527-r.png,train
BUS-BRA,bus_0528-l.png,test
BUS-BRA,bus_0528-r.png,test
BUS-BRA,bus_0529-l.png,train
BUS-BRA,bus_0529-r.png,train
BUS-BRA,bus_0530-l.png,train
BUS-BRA,bus_0530-r.png,train
BUS-BRA,bus_0531-l.png,test
BUS-BRA,bus_0531-r.png,test
BUS-BRA,bus_0532-l.png,test
BUS-BRA,bus_0532-r.png,test
BUS-BRA,bus_0533-l.png,test
BUS-BRA,bus_0533-r.png,test
BUS-BRA,bus_0534-l.png,test
BUS-BRA,bus_0534-r.png,test
BUS-BRA,bus_0535-l.png,val
BUS-BRA,bus_0535-r.png,val
BUS-BRA,bus_0536-l.png,train
BUS-BRA,bus_0536-r.png,train
BUS-BRA,bus_0537-l.png,train
BUS-BRA,bus_0537-r.png,train
BUS-BRA,bus_0538-l.png,train
BUS-BRA,bus_0538-r.png,train
BUS-BRA,bus_0539-l.png,train
BUS-BRA,bus_0539-r.png,train
BUS-BRA,bus_0540-l.png,train
BUS-BRA,bus_0540-r.png,train
BUS-BRA,bus_0541-l.png,train
BUS-BRA,bus_0541-r.png,train
BUS-BRA,bus_0542-l.png,train
BUS-BRA,bus_0542-r.png,train
BUS-BRA,bus_0543-l.png,test
BUS-BRA,bus_0543-r.png,test
BUS-BRA,bus_0544-l.png,test
BUS-BRA,bus_0544-r.png,test
BUS-BRA,bus_0545-l.png,test
BUS-BRA,bus_0545-r.png,test
BUS-BRA,bus_0546-l.png,train
BUS-BRA,bus_0546-r.png,train
BUS-BRA,bus_0547-l.png,val
BUS-BRA,bus_0547-r.png,val
BUS-BRA,bus_0548-l.png,train
BUS-BRA,bus_0548-r.png,train
BUS-BRA,bus_0549-l.png,train
BUS-BRA,bus_0549-r.png,train
BUS-BRA,bus_0550-l.png,train
BUS-BRA,bus_0550-r.png,train
BUS-BRA,bus_0551-l.png,train
BUS-BRA,bus_0551-r.png,train
BUS-BRA,bus_0552-l.png,train
BUS-BRA,bus_0552-r.png,train
BUS-BRA,bus_0553-l.png,test
BUS-BRA,bus_0553-r.png,test
BUS-BRA,bus_0554-l.png,train
BUS-BRA,bus_0554-r.png,train
BUS-BRA,bus_0555-l.png,train
BUS-BRA,bus_0555-r.png,train
BUS-BRA,bus_0556-l.png,train
BUS-BRA,bus_0556-r.png,train
BUS-BRA,bus_0557-l.png,train
BUS-BRA,bus_0557-r.png,train
BUS-BRA,bus_0558-l.png,test
BUS-BRA,bus_0558-r.png,test
BUS-BRA,bus_0559-l.png,val
BUS-BRA,bus_0559-r.png,val
BUS-BRA,bus_0560-l.png,test
BUS-BRA,bus_0560-r.png,test
BUS-BRA,bus_0561-l.png,train
BUS-BRA,bus_0561-r.png,train
BUS-BRA,bus_0562-l.png,train
BUS-BRA,bus_0562-r.png,train
BUS-BRA,bus_0563-l.png,test
BUS-BRA,bus_0563-r.png,test
BUS-BRA,bus_0564-l.png,train
BUS-BRA,bus_0564-r.png,train
BUS-BRA,bus_0565-l.png,val
BUS-BRA,bus_0565-r.png,val
BUS-BRA,bus_0566-l.png,train
BUS-BRA,bus_0566-r.png,train
BUS-BRA,bus_0567-l.png,train
BUS-BRA,bus_0567-r.png,train
BUS-BRA,bus_0568-l.png,test
BUS-BRA,bus_0568-r.png,test
BUS-BRA,bus_0569-l.png,train
BUS-BRA,bus_0569-r.png,train
BUS-BRA,bus_0570-l.png,train
BUS-BRA,bus_0570-r.png,train
BUS-BRA,bus_0571-l.png,train
BUS-BRA,bus_0571-r.png,train
BUS-BRA,bus_0572-l.png,test
BUS-BRA,bus_0572-r.png,test
BUS-BRA,bus_0573-l.png,test
BUS-BRA,bus_0573-r.png,test
BUS-BRA,bus_0574-l.png,val
BUS-BRA,bus_0574-r.png,val
BUS-BRA,bus_0575-l.png,train
BUS-BRA,bus_0575-r.png,train
BUS-BRA,bus_0576-l.png,test
BUS-BRA,bus_0576-r.png,test
BUS-BRA,bus_0577-l.png,test
BUS-BRA,bus_0577-r.png,test
BUS-BRA,bus_0578-l.png,train
BUS-BRA,bus_0578-r.png,train
BUS-BRA,bus_0579-l.png,train
BUS-BRA,bus_0579-r.png,train
BUS-BRA,bus_0580-l.png,test
BUS-BRA,bus_0580-r.png,test
BUS-BRA,bus_0581-l.png,train
BUS-BRA,bus_0581-r.png,train
BUS-BRA,bus_0582-l.png,train
BUS-BRA,bus_0582-r.png,train
BUS-BRA,bus_0583-l.png,val
BUS-BRA,bus_0583-r.png,val
BUS-BRA,bus_0584-l.png,train
BUS-BRA,bus_0584-r.png,train
BUS-BRA,bus_0585-l.png,train
BUS-BRA,bus_0585-r.png,train
BUS-BRA,bus_0586-l.png,train
BUS-BRA,bus_0586-r.png,train
BUS-BRA,bus_0587-l.png,train
BUS-BRA,bus_0587-r.png,train
BUS-BRA,bus_0588-l.png,test
BUS-BRA,bus_0588-r.png,test
BUS-BRA,bus_0589-l.png,train
BUS-BRA,bus_0589-r.png,train
BUS-BRA,bus_0590-l.png,test
BUS-BRA,bus_0590-r.png,test
BUS-BRA,bus_0591-l.png,test
BUS-BRA,bus_0591-r.png,test
BUS-BRA,bus_0592-l.png,val
BUS-BRA,bus_0592-r.png,val
BUS-BRA,bus_0593-l.png,train
BUS-BRA,bus_0593-r.png,train
BUS-BRA,bus_0594-l.png,train
BUS-BRA,bus_0594-r.png,train
BUS-BRA,bus_0595-l.png,train
BUS-BRA,bus_0595-r.png,train
BUS-BRA,bus_0596-l.png,train
BUS-BRA,bus_0596-r.png,train
BUS-BRA,bus_0597-l.png,test
BUS-BRA,bus_0597-r.png,test
BUS-BRA,bus_0598-l.png,train
BUS-BRA,bus_0598-r.png,train
BUS-BRA,bus_0599-l.png,test
BUS-BRA,bus_0599-r.png,test
BUS-BRA,bus_0600-l.png,test
BUS-BRA,bus_0600-r.png,test
BUS-BRA,bus_0601-l.png,train
BUS-BRA,bus_0601-r.png,train
BUS-BRA,bus_0602-l.png,train
BUS-BRA,bus_0602-r.png,train
BUS-BRA,bus_0603-l.png,train
BUS-BRA,bus_0603-r.png,train
BUS-BRA,bus_0604-l.png,val
BUS-BRA,bus_0604-r.png,val
BUS-BRA,bus_0605-l.png,train
BUS-BRA,bus_0605-r.png,train
BUS-BRA,bus_0606-l.png,train
BUS-BRA,bus_0606-r.png,train
BUS-BRA,bus_0607-l.png,train
BUS-BRA,bus_0607-r.png,train
BUS-BRA,bus_0608-l.png,test
BUS-BRA,bus_0608-r.png,test
BUS-BRA,bus_0609-l.png,train
BUS-BRA,bus_0609-r.png,train
BUS-BRA,bus_0610-l.png,test
BUS-BRA,bus_0610-r.png,test
BUS-BRA,bus_0611-l.png,test
BUS-BRA,bus_0611-r.png,test
BUS-BRA,bus_0612-l.png,val
BUS-BRA,bus_0612-r.png,val
BUS-BRA,bus_0613-l.png,train
BUS-BRA,bus_0613-r.png,train
BUS-BRA,bus_0614-l.png,train
BUS-BRA,bus_0614-r.png,train
BUS-BRA,bus_0615-l.png,train
BUS-BRA,bus_0615-r.png,train
BUS-BRA,bus_0616-l.png,train
BUS-BRA,bus_0616-r.png,train
BUS-BRA,bus_0617-l.png,train
BUS-BRA,bus_0617-r.png,train
BUS-BRA,bus_0618-l.png,train
BUS-BRA,bus_0618-r.png,train
BUS-BRA,bus_0619-l.png,test
BUS-BRA,bus_0619-r.png,test
BUS-BRA,bus_0620-l.png,train
BUS-BRA,bus_0620-r.png,train
BUS-BRA,bus_0621-l.png,val
BUS-BRA,bus_0621-r.png,val
BUS-BRA,bus_0622-l.png,train
BUS-BRA,bus_0622-r.png,train
BUS-BRA,bus_0623-l.png,test
BUS-BRA,bus_0623-r.png,test
BUS-BRA,bus_0624-l.png,test
BUS-BRA,bus_0624-r.png,test
BUS-BRA,bus_0625-l.png,test
BUS-BRA,bus_0625-r.png,test
BUS-BRA,bus_0626-l.png,test
BUS-BRA,bus_0626-r.png,test
BUS-BRA,bus_0627-l.png,test
BUS-BRA,bus_0627-r.png,test
BUS-BRA,bus_0628-l.png,train
BUS-BRA,bus_0628-r.png,train
BUS-BRA,bus_0629-l.png,train
BUS-BRA,bus_0629-r.png,train
BUS-BRA,bus_0630-l.png,train
BUS-BRA,bus_0630-r.png,train
BUS-BRA,bus_0631-l.png,train
BUS-BRA,bus_0631-r.png,train
BUS-BRA,bus_0632-l.png,train
BUS-BRA,bus_0632-r.png,train
BUS-BRA,bus_0633-l.png,test
BUS-BRA,bus_0633-r.png,test
BUS-BRA,bus_0634-l.png,test
BUS-BRA,bus_0634-r.png,test
BUS-BRA,bus_0635-l.png,train
BUS-BRA,bus_0635-r.png,train
BUS-BRA,bus_0636-l.png,train
BUS-BRA,bus_0636-r.png,train
BUS-BRA,bus_0637-l.png,val
BUS-BRA,bus_0637-r.png,val
BUS-BRA,bus_0638-l.png,test
BUS-BRA,bus_0638-r.png,test
BUS-BRA,bus_0639-l.png,train
BUS-BRA,bus_0639-r.png,train
BUS-BRA,bus_0640-l.png,train
BUS-BRA,bus_0640-r.png,train
BUS-BRA,bus_0641-l.png,train
BUS-BRA,bus_0641-r.png,train
BUS-BRA,bus_0642-l.png,train
BUS-BRA,bus_0642-r.png,train
BUS-BRA,bus_0643-l.png,train
BUS-BRA,bus_0643-r.png,train
BUS-BRA,bus_0644-l.png,val
BUS-BRA,bus_0644-r.png,val
BUS-BRA,bus_0645-l.png,train
BUS-BRA,bus_0645-r.png,train
BUS-BRA,bus_0646-l.png,train
BUS-BRA,bus_0646-r.png,train
BUS-BRA,bus_0647-l.png,test
BUS-BRA,bus_0647-r.png,test
BUS-BRA,bus_0648-l.png,test
BUS-BRA,bus_0648-r.png,test
BUS-BRA,bus_0649-l.png,test
BUS-BRA,bus_0649-r.png,test
BUS-BRA,bus_0650-l.png,val
BUS-BRA,bus_0650-r.png,val
BUS-BRA,bus_0651-l.png,train
BUS-BRA,bus_0651-r.png,train
BUS-BRA,bus_0652-l.png,train
BUS-BRA,bus_0652-r.png,train
BUS-BRA,bus_0653-l.png,train
BUS-BRA,bus_0653-r.png,train
BUS-BRA,bus_0654-l.png,train
BUS-BRA,bus_0654-r.png,train
BUS-BRA,bus_0655-l.png,train
BUS-BRA,bus_0655-r.png,train
BUS-BRA,bus_0656-l.png,train
BUS-BRA,bus_0656-r.png,train
BUS-BRA,bus_0657-l.png,train
BUS-BRA,bus_0657-r.png,train
BUS-BRA,bus_0658-s.png,train
BUS-BRA,bus_0659-l.png,test
BUS-BRA,bus_0659-r.png,test
BUS-BRA,bus_0660-l.png,test
BUS-BRA,bus_0660-r.png,test
BUS-BRA,bus_0661-l.png,test
BUS-BRA,bus_0661-r.png,test
BUS-BRA,bus_0662-l.png,val
BUS-BRA,bus_0662-r.png,val
BUS-BRA,bus_0663-l.png,train
BUS-BRA,bus_0663-r.png,train
BUS-BRA,bus_0664-s.png,test
BUS-BRA,bus_0665-l.png,test
BUS-BRA,bus_0665-r.png,test
BUS-BRA,bus_0666-l.png,test
BUS-BRA,bus_0666-r.png,test
BUS-BRA,bus_0667-l.png,val
BUS-BRA,bus_0667-r.png,val
BUS-BRA,bus_0668-l.png,train
BUS-BRA,bus_0668-r.png,train
BUS-BRA,bus_0669-l.png,train
BUS-BRA,bus_0669-r.png,train
BUS-BRA,bus_0670-l.png,train
BUS-BRA,bus_0670-r.png,train
BUS-BRA,bus_0671-l.png,train
BUS-BRA,bus_0671-r.png,train
BUS-BRA,bus_0672-l.png,train
BUS-BRA,bus_0672-r.png,train
BUS-BRA,bus_0673-l.png,test
BUS-BRA,bus_0673-r.png,test
BUS-BRA,bus_0674-l.png,val
BUS-BRA,bus_0674-r.png,val
BUS-BRA,bus_0675-s.png,train
BUS-BRA,bus_0676-s.png,train
BUS-BRA,bus_0677-s.png,train
BUS-BRA,bus_0678-s.png,train
BUS-BRA,bus_0679-s.png,train
BUS-BRA,bus_0680-s.png,train
BUS-BRA,bus_0681-s.png,test
BUS-BRA,bus_0682-s.png,train
BUS-BRA,bus_0683-s.png,test
BUS-BRA,bus_0684-s.png,test
BUS-BRA,bus_0685-l.png,train
BUS-BRA,bus_0685-r.png,train
BUS-BRA,bus_0686-s.png,train
BUS-BRA,bus_0687-s.png,train
BUS-BRA,bus_0688-s.png,train
BUS-BRA,bus_0689-s.png,train
BUS-BRA,bus_0690-s.png,train
BUS-BRA,bus_0691-s.png,train
BUS-BRA,bus_0692-s.png,test
BUS-BRA,bus_0693-s.png,test
BUS-BRA,bus_0694-s.png,test
BUS-BRA,bus_0695-l.png,test
BUS-BRA,bus_0695-r.png,test
BUS-BRA,bus_0696-s.png,test
BUS-BRA,bus_0697-l.png,val
BUS-BRA,bus_0697-r.png,val
BUS-BRA,bus_0698-s.png,train
BUS-BRA,bus_0699-s.png,val
BUS-BRA,bus_0700-s.png,train
BUS-BRA,bus_0701-s.png,train
BUS-BRA,bus_0702-s.png,train
BUS-BRA,bus_0703-s.png,train
BUS-BRA,bus_0704-s.png,train
BUS-BRA,bus_0705-s.png,train
BUS-BRA,bus_0706-s.png,test
BUS-BRA,bus_0707-s.png,train
BUS-BRA,bus_0708-s.png,train
BUS-BRA,bus_0709-s.png,test
BUS-BRA,bus_0710-s.png,train
BUS-BRA,bus_0711-s.png,test
BUS-BRA,bus_0712-s.png,val
BUS-BRA,bus_0713-l.png,train
BUS-BRA,bus_0713-r.png,train
BUS-BRA,bus_0714-s.png,train
BUS-BRA,bus_0715-s.png,train
BUS-BRA,bus_0716-s.png,train
BUS-BRA,bus_0717-s.png,test
BUS-BRA,bus_0718-s.png,train
BUS-BRA,bus_0719-s.png,train
BUS-BRA,bus_0720-s.png,test
BUS-BRA,bus_0721-s.png,train
BUS-BRA,bus_0722-s.png,train
BUS-BRA,bus_0723-s.png,train
BUS-BRA,bus_0724-s.png,train
BUS-BRA,bus_0725-s.png,test
BUS-BRA,bus_0726-s.png,train
BUS-BRA,bus_0727-s.png,test
BUS-BRA,bus_0728-s.png,val
BUS-BRA,bus_0729-s.png,test
BUS-BRA,bus_0730-s.png,test
BUS-BRA,bus_0731-s.png,train
BUS-BRA,bus_0732-s.png,train
BUS-BRA,bus_0733-s.png,test
BUS-BRA,bus_0734-s.png,val
BUS-BRA,bus_0735-s.png,test
BUS-BRA,bus_0736-s.png,test
BUS-BRA,bus_0737-s.png,train
BUS-BRA,bus_0738-s.png,train
BUS-BRA,bus_0739-s.png,train
BUS-BRA,bus_0740-s.png,train
BUS-BRA,bus_0741-s.png,train
BUS-BRA,bus_0742-s.png,train
BUS-BRA,bus_0743-s.png,train
BUS-BRA,bus_0744-s.png,val
BUS-BRA,bus_0745-s.png,train
BUS-BRA,bus_0746-s.png,train
BUS-BRA,bus_0747-s.png,train
BUS-BRA,bus_0748-s.png,train
BUS-BRA,bus_0749-s.png,train
BUS-BRA,bus_0750-s.png,test
BUS-BRA,bus_0751-s.png,test
BUS-BRA,bus_0752-s.png,test
BUS-BRA,bus_0753-s.png,train
BUS-BRA,bus_0754-s.png,val
BUS-BRA,bus_0755-s.png,train
BUS-BRA,bus_0756-s.png,train
BUS-BRA,bus_0757-s.png,train
BUS-BRA,bus_0758-s.png,train
BUS-BRA,bus_0759-s.png,val
BUS-BRA,bus_0760-s.png,train
BUS-BRA,bus_0761-s.png,test
BUS-BRA,bus_0762-s.png,test
BUS-BRA,bus_0763-s.png,test
BUS-BRA,bus_0764-s.png,test
BUS-BRA,bus_0765-s.png,train
BUS-BRA,bus_0766-s.png,train
BUS-BRA,bus_0767-s.png,test
BUS-BRA,bus_0768-s.png,train
BUS-BRA,bus_0769-s.png,train
BUS-BRA,bus_0770-s.png,train
BUS-BRA,bus_0771-s.png,train
BUS-BRA,bus_0772-s.png,train
BUS-BRA,bus_0773-s.png,test
BUS-BRA,bus_0774-s.png,train
BUS-BRA,bus_0775-s.png,train
BUS-BRA,bus_0776-s.png,test
BUS-BRA,bus_0777-s.png,test
BUS-BRA,bus_0778-s.png,train
BUS-BRA,bus_0779-s.png,test
BUS-BRA,bus_0780-s.png,val
BUS-BRA,bus_0781-s.png,train
BUS-BRA,bus_0782-s.png,train
BUS-BRA,bus_0783-s.png,val
BUS-BRA,bus_0784-s.png,train
BUS-BRA,bus_0785-s.png,train
BUS-BRA,bus_0786-s.png,train
BUS-BRA,bus_0787-s.png,train
BUS-BRA,bus_0788-s.png,train
BUS-BRA,bus_0789-s.png,test
BUS-BRA,bus_0790-s.png,test
BUS-BRA,bus_0791-s.png,test
BUS-BRA,bus_0792-s.png,train
BUS-BRA,bus_0793-s.png,val
BUS-BRA,bus_0794-s.png,train
BUS-BRA,bus_0795-s.png,train
BUS-BRA,bus_0796-s.png,train
BUS-BRA,bus_0797-s.png,train
BUS-BRA,bus_0798-s.png,train
BUS-BRA,bus_0799-s.png,test
BUS-BRA,bus_0800-s.png,test
BUS-BRA,bus_0801-s.png,test
BUS-BRA,bus_0802-s.png,val
BUS-BRA,bus_0803-s.png,train
BUS-BRA,bus_0804-s.png,train
BUS-BRA,bus_0805-s.png,train
BUS-BRA,bus_0806-s.png,train
BUS-BRA,bus_0807-s.png,train
BUS-BRA,bus_0808-s.png,train
BUS-BRA,bus_0809-s.png,test
BUS-BRA,bus_0810-s.png,test
BUS-BRA,bus_0811-s.png,val
BUS-BRA,bus_0812-s.png,test
BUS-BRA,bus_0813-s.png,train
BUS-BRA,bus_0814-s.png,train
BUS-BRA,bus_0815-s.png,test
BUS-BRA,bus_0816-s.png,val
BUS-BRA,bus_0817-s.png,train
BUS-BRA,bus_0818-s.png,test
BUS-BRA,bus_0819-s.png,test
BUS-BRA,bus_0820-s.png,train
BUS-BRA,bus_0821-s.png,train
BUS-BRA,bus_0822-s.png,train
BUS-BRA,bus_0823-s.png,train
BUS-BRA,bus_0824-s.png,train
BUS-BRA,bus_0825-s.png,val
BUS-BRA,bus_0826-s.png,test
BUS-BRA,bus_0827-s.png,test
BUS-BRA,bus_0828-s.png,train
BUS-BRA,bus_0829-s.png,test
BUS-BRA,bus_0830-s.png,train
BUS-BRA,bus_0831-s.png,train
BUS-BRA,bus_0832-s.png,train
BUS-BRA,bus_0833-s.png,train
BUS-BRA,bus_0834-s.png,train
BUS-BRA,bus_0835-s.png,train
BUS-BRA,bus_0836-s.png,train
BUS-BRA,bus_0837-s.png,val
BUS-BRA,bus_0838-s.png,train
BUS-BRA,bus_0839-s.png,train
BUS-BRA,bus_0840-s.png,test
BUS-BRA,bus_0841-s.png,train
BUS-BRA,bus_0842-s.png,test
BUS-BRA,bus_0843-s.png,train
BUS-BRA,bus_0844-s.png,train
BUS-BRA,bus_0845-s.png,train
BUS-BRA,bus_0846-s.png,test
BUS-BRA,bus_0847-s.png,train
BUS-BRA,bus_0848-l.png,test
BUS-BRA,bus_0848-r.png,test
BUS-BRA,bus_0849-l.png,test
BUS-BRA,bus_0849-r.png,test
BUS-BRA,bus_0850-l.png,val
BUS-BRA,bus_0850-r.png,val
BUS-BRA,bus_0851-s.png,test
BUS-BRA,bus_0852-s.png,train
BUS-BRA,bus_0853-l.png,train
BUS-BRA,bus_0853-r.png,train
BUS-BRA,bus_0854-s.png,train
BUS-BRA,bus_0855-l.png,train
BUS-BRA,bus_0855-r.png,train
BUS-BRA,bus_0856-l.png,test
BUS-BRA,bus_0856-r.png,test
BUS-BRA,bus_0857-l.png,train
BUS-BRA,bus_0857-r.png,train
BUS-BRA,bus_0858-l.png,train
BUS-BRA,bus_0858-r.png,train
BUS-BRA,bus_0859-l.png,test
BUS-BRA,bus_0859-r.png,test
BUS-BRA,bus_0860-l.png,train
BUS-BRA,bus_0860-r.png,train
BUS-BRA,bus_0861-l.png,test
BUS-BRA,bus_0861-r.png,test
BUS-BRA,bus_0862-l.png,val
BUS-BRA,bus_0862-r.png,val
BUS-BRA,bus_0863-l.png,train
BUS-BRA,bus_0863-r.png,train
BUS-BRA,bus_0864-l.png,train
BUS-BRA,bus_0864-r.png,train
BUS-BRA,bus_0865-l.png,train
BUS-BRA,bus_0865-r.png,train
BUS-BRA,bus_0866-l.png,train
BUS-BRA,bus_0866-r.png,train
BUS-BRA,bus_0867-l.png,train
BUS-BRA,bus_0867-r.png,train
BUS-BRA,bus_0868-l.png,train
BUS-BRA,bus_0868-r.png,train
BUS-BRA,bus_0869-l.png,test
BUS-BRA,bus_0869-r.png,test
BUS-BRA,bus_0870-l.png,test
BUS-BRA,bus_0870-r.png,test
BUS-BRA,bus_0871-s.png,test
BUS-BRA,bus_0872-l.png,val
BUS-BRA,bus_0872-r.png,val
BUS-BRA,bus_0873-l.png,train
BUS-BRA,bus_0873-r.png,train
BUS-BRA,bus_0874-l.png,train
BUS-BRA,bus_0874-r.png,train
BUS-BRA,bus_0875-l.png,train
BUS-BRA,bus_0875-r.png,train
BUS-BRA,bus_0876-l.png,train
BUS-BRA,bus_0876-r.png,train
BUS-BRA,bus_0877-l.png,train
BUS-BRA,bus_0877-r.png,train
BUS-BRA,bus_0878-s.png,train
BUS-BRA,bus_0879-l.png,train
BUS-BRA,bus_0879-r.png,train
BUS-BRA,bus_0880-l.png,test
BUS-BRA,bus_0880-r.png,test
BUS-BRA,bus_0881-l.png,test
BUS-BRA,bus_0881-r.png,test
BUS-BRA,bus_0882-l.png,train
BUS-BRA,bus_0882-r.png,train
BUS-BRA,bus_0883-s.png,train
BUS-BRA,bus_0884-l.png,train
BUS-BRA,bus_0884-r.png,train
BUS-BRA,bus_0885-l.png,test
BUS-BRA,bus_0885-r.png,test
BUS-BRA,bus_0886-s.png,train
BUS-BRA,bus_0887-l.png,train
BUS-BRA,bus_0887-r.png,train
BUS-BRA,bus_0888-l.png,test
BUS-BRA,bus_0888-r.png,test
BUS-BRA,bus_0889-l.png,test
BUS-BRA,bus_0889-r.png,test
BUS-BRA,bus_0890-l.png,val
BUS-BRA,bus_0890-r.png,val
BUS-BRA,bus_0891-l.png,train
BUS-BRA,bus_0891-r.png,train
BUS-BRA,bus_0892-l.png,train
BUS-BRA,bus_0892-r.png,train
BUS-BRA,bus_0893-l.png,test
BUS-BRA,bus_0893-r.png,test
BUS-BRA,bus_0894-s.png,test
BUS-BRA,bus_0895-l.png,train
BUS-BRA,bus_0895-r.png,train
BUS-BRA,bus_0896-l.png,test
BUS-BRA,bus_0896-r.png,test
BUS-BRA,bus_0897-l.png,train
BUS-BRA,bus_0897-r.png,train
BUS-BRA,bus_0898-l.png,val
BUS-BRA,bus_0898-r.png,val
BUS-BRA,bus_0899-s.png,train
BUS-BRA,bus_0900-l.png,test
BUS-BRA,bus_0900-r.png,test
BUS-BRA,bus_0901-l.png,train
BUS-BRA,bus_0901-r.png,train
BUS-BRA,bus_0902-l.png,train
BUS-BRA,bus_0902-r.png,train
BUS-BRA,bus_0903-l.png,train
BUS-BRA,bus_0903-r.png,train
BUS-BRA,bus_0904-l.png,train
BUS-BRA,bus_0904-r.png,train
BUS-BRA,bus_0905-l.png,val
BUS-BRA,bus_0905-r.png,val
BUS-BRA,bus_0906-l.png,test
BUS-BRA,bus_0906-r.png,test
BUS-BRA,bus_0907-l.png,test
BUS-BRA,bus_0907-r.png,test
BUS-BRA,bus_0908-l.png,train
BUS-BRA,bus_0908-r.png,train
BUS-BRA,bus_0909-l.png,train
BUS-BRA,bus_0909-r.png,train
BUS-BRA,bus_0910-l.png,train
BUS-BRA,bus_0910-r.png,train
BUS-BRA,bus_0911-l.png,test
BUS-BRA,bus_0911-r.png,test
BUS-BRA,bus_0912-l.png,train
BUS-BRA,bus_0912-r.png,train
BUS-BRA,bus_0913-l.png,val
BUS-BRA,bus_0913-r.png,val
BUS-BRA,bus_0914-l.png,train
BUS-BRA,bus_0914-r.png,train
BUS-BRA,bus_0915-l.png,train
BUS-BRA,bus_0915-r.png,train
BUS-BRA,bus_0916-l.png,train
BUS-BRA,bus_0916-r.png,train
BUS-BRA,bus_0917-s.png,train
BUS-BRA,bus_0918-l.png,train
BUS-BRA,bus_0918-r.png,train
BUS-BRA,bus_0919-l.png,test
BUS-BRA,bus_0919-r.png,test
BUS-BRA,bus_0920-s.png,test
BUS-BRA,bus_0921-l.png,test
BUS-BRA,bus_0921-r.png,test
BUS-BRA,bus_0922-l.png,test
BUS-BRA,bus_0922-r.png,test
BUS-BRA,bus_0923-l.png,train
BUS-BRA,bus_0923-r.png,train
BUS-BRA,bus_0924-l.png,train
BUS-BRA,bus_0924-r.png,train
BUS-BRA,bus_0925-l.png,test
BUS-BRA,bus_0925-r.png,test
BUS-BRA,bus_0926-l.png,val
BUS-BRA,bus_0926-r.png,val
BUS-BRA,bus_0927-l.png,train
BUS-BRA,bus_0927-r.png,train
BUS-BRA,bus_0928-l.png,train
BUS-BRA,bus_0928-r.png,train
BUS-BRA,bus_0929-l.png,test
BUS-BRA,bus_0929-r.png,test
BUS-BRA,bus_0930-l.png,train
BUS-BRA,bus_0930-r.png,train
BUS-BRA,bus_0931-l.png,train
BUS-BRA,bus_0931-r.png,train
BUS-BRA,bus_0932-l.png,train
BUS-BRA,bus_0932-r.png,train
BUS-BRA,bus_0933-l.png,val
BUS-BRA,bus_0933-r.png,val
BUS-BRA,bus_0934-l.png,train
BUS-BRA,bus_0934-r.png,train
BUS-BRA,bus_0935-l.png,train
BUS-BRA,bus_0935-r.png,train
BUS-BRA,bus_0936-l.png,train
BUS-BRA,bus_0936-r.png,train
BUS-BRA,bus_0937-l.png,test
BUS-BRA,bus_0937-r.png,test
BUS-BRA,bus_0938-l.png,val
BUS-BRA,bus_0938-r.png,val
BUS-BRA,bus_0939-s.png,test
BUS-BRA,bus_0940-s.png,test
BUS-BRA,bus_0941-l.png,train
BUS-BRA,bus_0941-r.png,train
BUS-BRA,bus_0942-l.png,train
BUS-BRA,bus_0942-r.png,train
BUS-BRA,bus_0943-s.png,val
BUS-BRA,bus_0944-l.png,train
BUS-BRA,bus_0944-r.png,train
BUS-BRA,bus_0945-l.png,train
BUS-BRA,bus_0945-r.png,train
BUS-BRA,bus_0946-s.png,test
BUS-BRA,bus_0947-l.png,train
BUS-BRA,bus_0947-r.png,train
BUS-BRA,bus_0948-l.png,train
BUS-BRA,bus_0948-r.png,train
BUS-BRA,bus_0949-l.png,test
BUS-BRA,bus_0949-r.png,test
BUS-BRA,bus_0950-l.png,train
BUS-BRA,bus_0950-r.png,train
BUS-BRA,bus_0951-l.png,train
BUS-BRA,bus_0951-r.png,train
BUS-BRA,bus_0952-l.png,test
BUS-BRA,bus_0952-r.png,test
BUS-BRA,bus_0953-l.png,train
BUS-BRA,bus_0953-r.png,train
BUS-BRA,bus_0954-l.png,train
BUS-BRA,bus_0954-r.png,train
BUS-BRA,bus_0955-l.png,train
BUS-BRA,bus_0955-r.png,train
BUS-BRA,bus_0956-l.png,test
BUS-BRA,bus_0956-r.png,test
BUS-BRA,bus_0957-s.png,test
BUS-BRA,bus_0958-l.png,train
BUS-BRA,bus_0958-r.png,train
BUS-BRA,bus_0959-s.png,val
BUS-BRA,bus_0960-l.png,train
BUS-BRA,bus_0960-r.png,train
BUS-BRA,bus_0961-l.png,train
BUS-BRA,bus_0961-r.png,train
BUS-BRA,bus_0962-l.png,train
BUS-BRA,bus_0962-r.png,train
BUS-BRA,bus_0963-l.png,train
BUS-BRA,bus_0963-r.png,train
BUS-BRA,bus_0964-l.png,test
BUS-BRA,bus_0964-r.png,test
BUS-BRA,bus_0965-l.png,test
BUS-BRA,bus_0965-r.png,test
BUS-BRA,bus_0966-l.png,train
BUS-BRA,bus_0966-r.png,train
BUS-BRA,bus_0967-l.png,train
BUS-BRA,bus_0967-r.png,train
BUS-BRA,bus_0968-l.png,train
BUS-BRA,bus_0968-r.png,train
BUS-BRA,bus_0969-l.png,train
BUS-BRA,bus_0969-r.png,train
BUS-BRA,bus_0970-s.png,test
BUS-BRA,bus_0971-l.png,test
BUS-BRA,bus_0971-r.png,test
BUS-BRA,bus_0972-s.png,val
BUS-BRA,bus_0973-l.png,train
BUS-BRA,bus_0973-r.png,train
BUS-BRA,bus_0974-l.png,train
BUS-BRA,bus_0974-r.png,train
BUS-BRA,bus_0975-l.png,train
BUS-BRA,bus_0975-r.png,train
BUS-BRA,bus_0976-l.png,train
BUS-BRA,bus_0976-r.png,train
BUS-BRA,bus_0977-l.png,train
BUS-BRA,bus_0977-r.png,train
BUS-BRA,bus_0978-l.png,train
BUS-BRA,bus_0978-r.png,train
BUS-BRA,bus_0979-l.png,test
BUS-BRA,bus_0979-r.png,test
BUS-BRA,bus_0980-l.png,test
BUS-BRA,bus_0980-r.png,test
BUS-BRA,bus_0981-l.png,test
BUS-BRA,bus_0981-r.png,test
BUS-BRA,bus_0982-l.png,train
BUS-BRA,bus_0982-r.png,train
BUS-BRA,bus_0983-l.png,test
BUS-BRA,bus_0983-r.png,test
BUS-BRA,bus_0984-l.png,test
BUS-BRA,bus_0984-r.png,test
BUS-BRA,bus_0985-l.png,test
BUS-BRA,bus_0985-r.png,test
BUS-BRA,bus_0986-l.png,train
BUS-BRA,bus_0986-r.png,train
BUS-BRA,bus_0987-l.png,val
BUS-BRA,bus_0987-r.png,val
BUS-BRA,bus_0988-s.png,train
BUS-BRA,bus_0989-l.png,train
BUS-BRA,bus_0989-r.png,train
BUS-BRA,bus_0990-l.png,val
BUS-BRA,bus_0990-r.png,val
BUS-BRA,bus_0991-s.png,train
BUS-BRA,bus_0992-l.png,train
BUS-BRA,bus_0992-r.png,train
BUS-BRA,bus_0993-l.png,train
BUS-BRA,bus_0993-r.png,train
BUS-BRA,bus_0994-l.png,train
BUS-BRA,bus_0994-r.png,train
BUS-BRA,bus_0995-l.png,test
BUS-BRA,bus_0995-r.png,test
BUS-BRA,bus_0996-l.png,train
BUS-BRA,bus_0996-r.png,train
BUS-BRA,bus_0997-l.png,test
BUS-BRA,bus_0997-r.png,test
BUS-BRA,bus_0998-s.png,train
BUS-BRA,bus_0999-l.png,test
BUS-BRA,bus_0999-r.png,test
BUS-BRA,bus_1000-l.png,train
BUS-BRA,bus_1000-r.png,train
BUS-BRA,bus_1001-l.png,val
BUS-BRA,bus_1001-r.png,val
BUS-BRA,bus_1002-l.png,train
BUS-BRA,bus_1002-r.png,train
BUS-BRA,bus_1003-l.png,train
BUS-BRA,bus_1003-r.png,train
BUS-BRA,bus_1004-l.png,train
BUS-BRA,bus_1004-r.png,train
BUS-BRA,bus_1005-s.png,train
BUS-BRA,bus_1006-l.png,train
BUS-BRA,bus_1006-r.png,train
BUS-BRA,bus_1007-l.png,train
BUS-BRA,bus_1007-r.png,train
BUS-BRA,bus_1008-l.png,train
BUS-BRA,bus_1008-r.png,train
BUS-BRA,bus_1009-l.png,train
BUS-BRA,bus_1009-r.png,train
BUS-BRA,bus_1010-l.png,test
BUS-BRA,bus_1010-r.png,test
BUS-BRA,bus_1011-l.png,test
BUS-BRA,bus_1011-r.png,test
BUS-BRA,bus_1012-l.png,test
BUS-BRA,bus_1012-r.png,test
BUS-BRA,bus_1013-l.png,test
BUS-BRA,bus_1013-r.png,test
BUS-BRA,bus_1014-l.png,train
BUS-BRA,bus_1014-r.png,train
BUS-BRA,bus_1015-s.png,test
BUS-BRA,bus_1016-l.png,train
BUS-BRA,bus_1016-r.png,train
BUS-BRA,bus_1017-l.png,val
BUS-BRA,bus_1017-r.png,val
BUS-BRA,bus_1018-l.png,train
BUS-BRA,bus_1018-r.png,train
BUS-BRA,bus_1019-l.png,train
BUS-BRA,bus_1019-r.png,train
BUS-BRA,bus_1020-l.png,train
BUS-BRA,bus_1020-r.png,train
BUS-BRA,bus_1021-l.png,train
BUS-BRA,bus_1021-r.png,train
BUS-BRA,bus_1022-l.png,test
BUS-BRA,bus_1022-r.png,test
BUS-BRA,bus_1023-l.png,test
BUS-BRA,bus_1023-r.png,test
BUS-BRA,bus_1024-l.png,val
BUS-BRA,bus_1024-r.png,val
BUS-BRA,bus_1025-l.png,train
BUS-BRA,bus_1025-r.png,train
BUS-BRA,bus_1026-l.png,test
BUS-BRA,bus_1026-r.png,test
BUS-BRA,bus_1027-l.png,test
BUS-BRA,bus_1027-r.png,test
BUS-BRA,bus_1028-l.png,train
BUS-BRA,bus_1028-r.png,train
BUS-BRA,bus_1029-l.png,train
BUS-BRA,bus_1029-r.png,train
BUS-BRA,bus_1030-l.png,train
BUS-BRA,bus_1030-r.png,train
BUS-BRA,bus_1031-l.png,train
BUS-BRA,bus_1031-r.png,train
BUS-BRA,bus_1032-l.png,train
BUS-BRA,bus_1032-r.png,train
BUS-BRA,bus_1033-l.png,val
BUS-BRA,bus_1033-r.png,val
BUS-BRA,bus_1034-l.png,train
BUS-BRA,bus_1034-r.png,train
BUS-BRA,bus_1035-l.png,train
BUS-BRA,bus_1035-r.png,train
BUS-BRA,bus_1036-l.png,train
BUS-BRA,bus_1036-r.png,train
BUS-BRA,bus_1037-l.png,train
BUS-BRA,bus_1037-r.png,train
BUS-BRA,bus_1038-l.png,train
BUS-BRA,bus_1038-r.png,train
BUS-BRA,bus_1039-l.png,train
BUS-BRA,bus_1039-r.png,train
BUS-BRA,bus_1040-l.png,test
BUS-BRA,bus_1040-r.png,test
BUS-BRA,bus_1041-l.png,train
BUS-BRA,bus_1041-r.png,train
BUS-BRA,bus_1042-l.png,test
BUS-BRA,bus_1042-r.png,test
BUS-BRA,bus_1043-l.png,train
BUS-BRA,bus_1043-r.png,train
BUS-BRA,bus_1044-l.png,test
BUS-BRA,bus_1044-r.png,test
BUS-BRA,bus_1045-l.png,val
BUS-BRA,bus_1045-r.png,val
BUS-BRA,bus_1046-l.png,train
BUS-BRA,bus_1046-r.png,train
BUS-BRA,bus_1047-l.png,train
BUS-BRA,bus_1047-r.png,train
BUS-BRA,bus_1048-l.png,train
BUS-BRA,bus_1048-r.png,train
BUS-BRA,bus_1049-l.png,train
BUS-BRA,bus_1049-r.png,train
BUS-BRA,bus_1050-l.png,test
BUS-BRA,bus_1050-r.png,test
BUS-BRA,bus_1051-l.png,train
BUS-BRA,bus_1051-r.png,train
BUS-BRA,bus_1052-l.png,test
BUS-BRA,bus_1052-r.png,test
BUS-BRA,bus_1053-s.png,test
BUS-BRA,bus_1054-l.png,test
BUS-BRA,bus_1054-r.png,test
BUS-BRA,bus_1055-l.png,val
BUS-BRA,bus_1055-r.png,val
BUS-BRA,bus_1056-l.png,test
BUS-BRA,bus_1056-r.png,test
BUS-BRA,bus_1057-l.png,train
BUS-BRA,bus_1057-r.png,train
BUS-BRA,bus_1058-l.png,train
BUS-BRA,bus_1058-r.png,train
BUS-BRA,bus_1059-l.png,train
BUS-BRA,bus_1059-r.png,train
BUS-BRA,bus_1060-s.png,test
BUS-BRA,bus_1061-l.png,train
BUS-BRA,bus_1061-r.png,train
BUS-BRA,bus_1062-l.png,train
BUS-BRA,bus_1062-r.png,train
BUS-BRA,bus_1063-l.png,train
BUS-BRA,bus_1063-r.png,train
BUS-BRA,bus_1064-l.png,test
BUS-BRA,bus_1064-r.png,test
'''

## 1. Load the scans and masks, with the classifier's split

In [ ]:
rows = []
busi_root = glob.glob(f"{INPUT}/**/Dataset_BUSI_with_GT", recursive=True)[0]
for label in ["normal", "benign", "malignant"]:
    for path in sorted(glob.glob(f"{busi_root}/{label}/*.png")):
        if "_mask" not in os.path.basename(path):
            rows.append({"path": path, "source": "BUSI", "label": label,
                         "masks": sorted(glob.glob(glob.escape(path[:-4]) + "_mask*.png"))})

TCIA = "https://www.cancerimagingarchive.net/wp-content/uploads/"
zip_path = download(TCIA + "BrEaST-Lesions_USG-images_and_masks-Dec-15-2023.zip", f"{TMP}/breast_usg.zip")
xlsx_path = download(TCIA + "BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx", f"{TMP}/breast_usg.xlsx")
if not glob.glob(f"{TMP}/breast_usg/**/case001.png", recursive=True):
    zipfile.ZipFile(zip_path).extractall(f"{TMP}/breast_usg")
img_dir = os.path.dirname(glob.glob(f"{TMP}/breast_usg/**/case001.png", recursive=True)[0])
for r in pd.read_excel(xlsx_path).itertuples():
    masks = [f"{img_dir}/{m}" for m in r.Mask_tumor_filename.split("&")] if isinstance(r.Mask_tumor_filename, str) else []
    rows.append({"path": f"{img_dir}/{r.Image_filename}", "source": "BrEaST", "label": r.Classification, "masks": masks})

bra_dir = os.path.dirname(glob.glob(f"{INPUT}/**/bus_data.csv", recursive=True)[0])
for r in pd.read_csv(f"{bra_dir}/bus_data.csv").itertuples():
    mask = f"{bra_dir}/Masks/mask_{r.ID[4:]}.png"
    rows.append({"path": f"{bra_dir}/Images/{r.ID}.png", "source": "BUS-BRA", "label": r.Pathology,
                 "masks": [mask] if os.path.exists(mask) else []})

df = pd.DataFrame(rows)
split = pd.read_csv(io.StringIO(SPLIT_CSV))
split = dict(zip(split["source"] + "/" + split["file"], split["split"]))
df["split"] = (df["source"] + "/" + df["path"].map(os.path.basename)).map(split)
df = df[df["split"].notna()]                                   # the classifier dropped these (near-duplicates)
# a lesion scan without a mask can't be used; normal scans get an empty mask
df = df[(df["label"] == "normal") | (df["masks"].map(len) > 0)].reset_index(drop=True)
print(pd.crosstab([df["source"], df["label"]], df["split"], margins=True))

grays, masks = [], []
for r in df.itertuples():
    img = Image.open(r.path)
    grays.append(np.asarray(to_gray(img)))
    masks.append(to_mask(r.masks, img.size) if r.masks else np.zeros((IMG, IMG), bool))
grays, masks = np.stack(grays), np.stack(masks)
idx = {s: np.where(df["split"] == s)[0] for s in ["train", "val", "test"]}
print("median share of the image covered by a lesion:", np.median(masks[(df["label"] != "normal").values].mean((1, 2))).round(3))

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, i in zip(axes.flat, df[df["label"] != "normal"].groupby("source").head(4).index):
    ax.imshow(grays[i], cmap="gray"); ax.contour(masks[i], colors="lime", linewidths=1)
    ax.set_title(f"{df.source[i]} · {df.label[i]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Training

In [ ]:
AUG = v2.Compose([
    v2.RandomHorizontalFlip(),   # no vertical flip: the transducer is always at the top of the image
    v2.RandomApply([v2.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.8, 1.2))], p=0.7),
    v2.RandomApply([v2.ColorJitter(brightness=0.35, contrast=0.35)], p=0.7),
    v2.RandomApply([v2.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.25),
])

class SegDS(Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = np.asarray(ids), augment
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        g, m = grays[j], masks[j]
        if self.augment:
            gi, mi = AUG(tv_tensors.Image(torch.from_numpy(g.copy())[None]),
                         tv_tensors.Mask(torch.from_numpy(m.astype(np.uint8))[None]))
            g, m = gi[0].numpy(), mi[0].numpy() > 0
        return torch.from_numpy(normalize(g)), torch.from_numpy(m[None].astype(np.float32)), int(j)

def loader(ids, augment=False):
    return DataLoader(SegDS(ids, augment), batch_size=BATCH, shuffle=augment, drop_last=augment, num_workers=4,
                      pin_memory=True)

net = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
dice_loss = smp.losses.DiceLoss("binary", from_logits=True)
bce = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=EPOCHS * (len(idx["train"]) // BATCH), pct_start=0.1)
scaler = torch.cuda.amp.GradScaler()

@torch.no_grad()
def predict(model, ids):
    model.eval()
    out = []
    for x, *_ in loader(ids):
        with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.sigmoid(model(x.to(DEVICE)).float()).cpu().numpy()[:, 0])
    return np.concatenate(out)

def dice(pred, true):
    # an empty outline on an empty mask counts as perfect
    inter, total = (pred & true).sum(), pred.sum() + true.sum()
    return 1.0 if total == 0 else 2 * inter / total

def iou(pred, true):
    union = (pred | true).sum()
    return 1.0 if union == 0 else (pred & true).sum() / union

lesion_val = [k for k, j in enumerate(idx["val"]) if masks[j].any()]
best, history = (-1, None, -1), []
for epoch in range(EPOCHS):
    net.train()
    losses = []
    for x, y, _ in loader(idx["train"], augment=True):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits = net(x)
            loss = bce(logits.float(), y) + dice_loss(logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        losses.append(loss.item())
    pv = predict(net, idx["val"]) > 0.5
    val_dice = float(np.mean([dice(pv[k], masks[idx["val"][k]]) for k in lesion_val]))
    history.append((epoch, float(np.mean(losses)), val_dice))
    print(f"epoch {epoch:2d}  loss {np.mean(losses):.4f}  val Dice (lesions) {val_dice:.4f}")
    if val_dice > best[0]:
        best = (val_dice, copy.deepcopy(net.state_dict()), epoch)
net.load_state_dict(best[1])
print(f"best epoch {best[2]}, val Dice {best[0]:.4f}")

h = np.array(history)
plt.figure(figsize=(7, 3.5)); plt.plot(h[:, 0], h[:, 1], label="train loss"); plt.plot(h[:, 0], h[:, 2], label="val Dice")
plt.axvline(best[2], ls="--", c="grey"); plt.legend(); plt.xlabel("epoch"); plt.tight_layout()
plt.savefig(f"{OUT}/seg_training.png", dpi=130); plt.show()

## 3. Export to ONNX (fp16 weight storage) and check parity

The backend runs this ONNX file. As with the classifier, conv weights are *stored* in fp16 and cast back to fp32 when the
model loads, halving the file size; all computation stays fp32.

In [ ]:
import onnx, onnxruntime as ort
from onnx import helper, numpy_helper, TensorProto

net = net.float().cpu().eval()
fp32_path = f"{TMP}/breast_seg_fp32.onnx"
dummy = torch.from_numpy(normalize(grays[0]))[None]
export_args = dict(input_names=["image"], output_names=["mask_logits"], opset_version=17,
                   dynamic_axes={"image": {0: "batch"}, "mask_logits": {0: "batch"}})
try:
    torch.onnx.export(net, dummy, fp32_path, dynamo=False, **export_args)
except TypeError:
    torch.onnx.export(net, dummy, fp32_path, **export_args)

model = onnx.load(fp32_path)
graph = model.graph
initializers, casts = [], []
for init in graph.initializer:
    w = numpy_helper.to_array(init)
    if w.dtype == np.float32 and w.ndim >= 2:
        initializers.append(numpy_helper.from_array(w.astype(np.float16), init.name + "_fp16"))
        casts.append(helper.make_node("Cast", [init.name + "_fp16"], [init.name], to=TensorProto.FLOAT))
    else:
        initializers.append(numpy_helper.from_array(w, init.name))
nodes = casts + [copy.deepcopy(n) for n in graph.node]
graph.ClearField("initializer"); graph.initializer.extend(initializers)
graph.ClearField("node"); graph.node.extend(nodes)
onnx.checker.check_model(model)
ONNX_PATH = f"{OUT}/breast_seg_unet.onnx"
onnx.save(model, ONNX_PATH)
print(f"fp32 {os.path.getsize(fp32_path) / 1e6:.1f} MB -> deployed {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB")

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
def onnx_prob(ids, batch=16):
    out = [session.run(None, {"image": np.stack([normalize(grays[j]) for j in ids[i:i + batch]])})[0][:, 0]
           for i in range(0, len(ids), batch)]
    return 1 / (1 + np.exp(-np.concatenate(out)))

with torch.no_grad():
    torch_prob = torch.sigmoid(net(torch.from_numpy(np.stack([normalize(grays[j]) for j in idx["test"][:32]])))).numpy()[:, 0]
parity = float(np.abs(onnx_prob(idx["test"][:32]) - torch_prob).max())
print("max |P_onnx - P_torch| on 32 test scans:", round(parity, 5))
assert parity < 0.05

## 4. Post-processing and the test set

The backend keeps pixels with probability ≥ 0.5, then only the **largest connected region** (one lesion per result), and
drops outlines smaller than a minimum area. The minimum area is chosen on the **validation** split and then applied unchanged
to the test split.

In [ ]:
def largest_component(m):
    # 4-connected labelling without scipy, mirroring backend/app.py
    lab = np.zeros(m.shape, np.int32)
    best_label, best_n, cur = 0, 0, 0
    for y0, x0 in zip(*np.nonzero(m)):
        if lab[y0, x0]:
            continue
        cur += 1
        stack, n = [(y0, x0)], 0
        lab[y0, x0] = cur
        while stack:
            y, x = stack.pop(); n += 1
            for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= yy < m.shape[0] and 0 <= xx < m.shape[1] and m[yy, xx] and not lab[yy, xx]:
                    lab[yy, xx] = cur; stack.append((yy, xx))
        if n > best_n:
            best_label, best_n = cur, n
    return lab == best_label if best_label else np.zeros_like(m)

def postprocess(prob, min_area):
    m = largest_component(prob >= 0.5)
    return m if m.mean() >= min_area else np.zeros_like(m)

def evaluate(ids, prob, min_area):
    out = []
    for j, p in zip(ids, prob):
        pred, true = postprocess(p, min_area), masks[j]
        out.append({"source": df.source[j], "label": df.label[j], "dice": dice(pred, true), "iou": iou(pred, true),
                    "outlined": bool(pred.any())})
    return pd.DataFrame(out)

prob_val, prob_test = onnx_prob(idx["val"]), onnx_prob(idx["test"])
choices = []
for min_area in [0.0, 0.002, 0.005, 0.01]:
    ev = evaluate(idx["val"], prob_val, min_area)
    les, nor = ev[ev.label != "normal"], ev[ev.label == "normal"]
    choices.append((min_area, les.dice.mean(), 1 - nor.outlined.mean()))
    print(f"min area {min_area:.3f}: val lesion Dice {les.dice.mean():.4f}, normal scans left blank {1 - nor.outlined.mean():.2%}")
top = max(c[1] for c in choices)
MIN_AREA = max(c[0] for c in choices if c[1] >= top - 0.005)   # the largest minimum area that costs < 0.005 Dice
print("chosen minimum area:", MIN_AREA)

ev = evaluate(idx["test"], prob_test, MIN_AREA)
les, nor = ev[ev.label != "normal"], ev[ev.label == "normal"]
def summary(e):
    return {"scans": int(len(e)), "dice_mean": round(float(e.dice.mean()), 4), "dice_median": round(float(e.dice.median()), 4),
            "iou_mean": round(float(e.iou.mean()), 4), "share_dice_at_least_0_5": round(float((e.dice >= 0.5).mean()), 4),
            "share_missed": round(float((~e.outlined).mean()), 4)}
test_metrics = summary(les)
per_source = {s: summary(g) for s, g in les.groupby("source")}
per_label = {s: summary(g) for s, g in les.groupby("label")}
normal_blank = {"scans": int(len(nor)), "share_left_blank": round(float(1 - nor.outlined.mean()), 4)}
print(json.dumps({"test_lesions": test_metrics, "per_source": per_source, "per_label": per_label,
                  "normal_scans": normal_blank}, indent=2))

# qualitative examples: best, typical and worst test outlines
order = les.sort_values("dice").index.tolist()
mid = len(order) // 2
pick = order[-4:] + order[mid - 2: mid + 2] + order[:4]
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, k in zip(axes.flat, pick):
    j = idx["test"][k]
    ax.imshow(grays[j], cmap="gray")
    ax.contour(masks[j], colors="lime", linewidths=1.2)
    pred = postprocess(prob_test[k], MIN_AREA)
    if pred.any():
        ax.contour(pred, colors="magenta", linewidths=1.2)
    ax.set_title(f"{df.source[j]} · {df.label[j]} · Dice {ev.dice[k]:.2f}", fontsize=9); ax.axis("off")
plt.suptitle("Test scans: radiologist (green) vs model (magenta); rows = best, typical, worst")
plt.tight_layout(); plt.savefig(f"{OUT}/seg_examples.png", dpi=130); plt.show()

In [ ]:
meta = {
    "onnx_file": "breast_seg_unet.onnx",
    "architecture": "U-Net, ResNet34 encoder (ImageNet), segmentation_models_pytorch " + smp.__version__,
    "input": {"size": IMG, "mean": MEAN.ravel().tolist(), "std": STD.ravel().tolist(), "channels": "grayscale x3",
              "preprocessing": "grayscale, padded to a square, bilinear resize"},
    "probability_threshold": 0.5,
    "min_area_fraction": MIN_AREA,
    "best_epoch": int(best[2]), "epochs": EPOCHS, "val_dice": round(float(best[0]), 4),
    "split": "same as the deployed classifier (ml/busbra_split.csv)",
    "train_scans": int(len(idx["train"])), "val_scans": int(len(idx["val"])), "test_scans": int(len(idx["test"])),
    "test_metrics": test_metrics, "per_source_test": per_source, "per_label_test": per_label,
    "normal_test_scans": normal_blank,
    "onnx_parity": round(parity, 5),
    "datasets": ["BUSI (Al-Dhabyani et al., 2020)", "BrEaST-Lesions-USG (Pawłowska et al., 2024, TCIA, CC BY 4.0)",
                 "BUS-BRA (Gómez-Flores et al., Medical Physics 2024)"],
}
with open(f"{OUT}/breast_seg_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(os.listdir(OUT))